# MixLLM 4/8/16 real T4 gate

This notebook embeds the current Python and CUDA sources and validates them on NVIDIA T4 / SM75. It runs model_gate import/allocator checks and native operator benchmarks for mixed and pure precision partitions. Operator timings are not model throughput. Full-model Qwen quality remains not_run unless separately measured.


In [ ]:
import hashlib, json, os, platform, subprocess, sys
from pathlib import Path
import torch
ARTIFACT_DIR = Path('/kaggle/working')
print('Python', sys.version)
print('STARTUP_HEARTBEAT', flush=True); print('PyTorch', torch.__version__, flush=True); print('DEVICE_COUNT', torch.cuda.device_count(), flush=True); print('ACTIVE_DEVICE', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, flush=True)


In [ ]:
sources = {'mixllm/__init__.py': 'import sysconfig\nfrom pathlib import Path\n\n\ndef load_cuda_extension() -> None:\n    """Load the compiled extension only when the CUDA runtime is requested."""\n    import torch\n\n    current_dir = Path(__file__).resolve().parent\n    ext_suffix = sysconfig.get_config_var("EXT_SUFFIX")\n    extension = current_dir / f"kernels{ext_suffix}"\n    if not extension.is_file():\n        raise RuntimeError(\n            f"MixLLM CUDA extension is not built: {extension}. "\n            "Reference quantization and capability modules remain available."\n        )\n    torch.ops.load_library(str(extension))\n\n\ndef __getattr__(name):\n    """Preserve the legacy package API without eager CUDA side effects."""\n    if name == "LinearMixLLM":\n        load_cuda_extension()\n        from mixllm.nn.modules.linear import LinearMixLLM\n        return LinearMixLLM\n    if name == "LinearMixLLM4vLLM":\n        load_cuda_extension()\n        from mixllm.nn.modules.linear_for_vllm import LinearMixLLM4vLLM\n        return LinearMixLLM4vLLM\n    if name == "MixLLMConfig":\n        from mixllm.nn.modules.mixllm_config import MixLLMConfig\n        return MixLLMConfig\n    raise AttributeError(name)\n', 'mixllm/quantization/__init__.py': '', 'mixllm/nn/__init__.py': '', 'mixllm/nn/modules/__init__.py': '', 'mixllm/quantization/three_level.py': '"""Three-level allocation and reference quantization for MixLLM.\n\nThe allocator uses the value of an upgrade (L4-L8 or L8-L16), rather than\nthe absolute loss of the lowest precision.  This keeps the existing global\noutput-feature design while making the extension to FP16 unambiguous.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport json\nimport math\nfrom pathlib import Path\nfrom typing import Dict, Mapping, Sequence\n\nimport torch\n\n\nLEVELS = (4, 8, 16)\n\n\n@dataclass(frozen=True)\nclass ThreeLevelBudget:\n    """Percentages of output channels assigned to each precision."""\n\n    bit4: int\n    bit8: int\n    bit16: int\n\n    def __post_init__(self) -> None:\n        values = (self.bit4, self.bit8, self.bit16)\n        if any(value < 0 for value in values) or sum(values) != 100:\n            raise ValueError("bit4 + bit8 + bit16 must equal 100")\n\n    def as_dict(self) -> Dict[int, int]:\n        return {4: self.bit4, 8: self.bit8, 16: self.bit16}\n\n\n@dataclass(frozen=True)\nclass ThreeLevelAllocation:\n    """A complete, deterministic output-channel allocation."""\n\n    indices: Dict[int, tuple[int, ...]]\n    scores: Dict[int, tuple[float, ...]]\n    budget: ThreeLevelBudget\n    version: int = 1\n    metadata: Dict[str, object] = field(default_factory=dict)\n\n    def verify(self, out_features: int) -> None:\n        assigned = [index for bit in LEVELS for index in self.indices[bit]]\n        if sorted(assigned) != list(range(out_features)):\n            raise ValueError("allocation must cover every output channel exactly once")\n        if any(index < 0 or index >= out_features for index in assigned):\n            raise ValueError("allocation contains an out-of-range channel")\n\n    def to_json(self, path: str | Path) -> None:\n        payload = {\n            "format": "mixllm-three-level",\n            "version": self.version,\n            "metadata": self.metadata,\n            "budget": {str(k): v for k, v in self.budget.as_dict().items()},\n            "indices": {str(k): list(v) for k, v in self.indices.items()},\n            "scores": {str(k): list(v) for k, v in self.scores.items()},\n        }\n        Path(path).write_text(json.dumps(payload, indent=2) + "\\n", encoding="utf-8")\n\n    @classmethod\n    def from_json(cls, path: str | Path) -> "ThreeLevelAllocation":\n        payload = json.loads(Path(path).read_text(encoding="utf-8"))\n        if payload.get("format") != "mixllm-three-level":\n            raise ValueError("unsupported allocation format")\n        budget = ThreeLevelBudget(**{\n            f"bit{bit}": int(value) for bit, value in payload["budget"].items()\n        })\n        allocation = cls(\n            indices={int(bit): tuple(int(i) for i in values)\n                     for bit, values in payload["indices"].items()},\n            scores={int(bit): tuple(float(i) for i in values)\n                    for bit, values in payload["scores"].items()},\n            budget=budget,\n            version=int(payload["version"]),\n            metadata=dict(payload.get("metadata", {})),\n        )\n        if set(allocation.indices) != set(LEVELS):\n            raise ValueError("allocation must contain 4, 8 and 16 bit partitions")\n        return allocation\n\n\ndef _rounded_counts(out_features: int, budget: ThreeLevelBudget,\n                    alignment: int) -> Dict[int, int]:\n    """Round counts to alignment while preserving the requested total."""\n    raw = {bit: out_features * pct / 100 for bit, pct in budget.as_dict().items()}\n    counts = {bit: int(raw[bit] // alignment) * alignment for bit in LEVELS}\n    remainder = out_features - sum(counts.values())\n    order = sorted(LEVELS, key=lambda bit: (raw[bit] - counts[bit], -bit), reverse=True)\n    for bit in order:\n        if remainder < alignment:\n            break\n        counts[bit] += alignment\n        remainder -= alignment\n    # A tail is legal for the metadata/reference path; the backend may reject it.\n    counts[order[0]] += remainder\n    return counts\n\n\ndef allocate_channels(losses: Mapping[int, torch.Tensor | Sequence[float]],\n                      budget: ThreeLevelBudget, alignment: int = 1\n                      ) -> ThreeLevelAllocation:\n    """Allocate channels using marginal upgrade losses.\n\n    ``losses[bit][channel]`` is the estimated output error when that channel\n    is quantized at ``bit``.  Lower is better.  The allocation is global across\n    all channels and therefore matches MixLLM\'s output-feature abstraction.\n    """\n    if alignment < 1:\n        raise ValueError("alignment must be positive")\n    values = {bit: torch.as_tensor(losses[bit], dtype=torch.float64).flatten()\n              for bit in LEVELS}\n    lengths = {tensor.numel() for tensor in values.values()}\n    if len(lengths) != 1 or next(iter(lengths), 0) == 0:\n        raise ValueError("all loss vectors must have the same non-zero length")\n    if any(not torch.isfinite(tensor).all() for tensor in values.values()):\n        raise ValueError("loss vectors must be finite")\n    out_features = next(iter(lengths))\n    counts = _rounded_counts(out_features, budget, alignment)\n\n    # Upgrades are ranked by the error removed by moving up one level.\n    benefit8 = values[4] - values[8]\n    benefit16 = values[8] - values[16]\n    # Choose the final classes hierarchically.  An FP16 channel is upgraded\n    # from INT8, so it must be selected before INT8 candidates are considered.\n    selected: Dict[int, list[int]] = {4: [], 8: [], 16: []}\n    selected[16] = [index for _, index in sorted(\n        ((float(benefit16[i]), i) for i in range(out_features)),\n        key=lambda item: (-item[0], item[1]))[:counts[16]]]\n    used = set(selected[16])\n    selected[8] = [index for _, index in sorted(\n        ((float(benefit8[i]), i) for i in range(out_features) if i not in used),\n        key=lambda item: (-item[0], item[1]))[:counts[8]]]\n    used.update(selected[8])\n    selected[4] = [index for index in range(out_features) if index not in used]\n    if len(selected[4]) != counts[4]:\n        raise ValueError("budget/alignment could not be represented exactly")\n    result = ThreeLevelAllocation(\n        indices={bit: tuple(sorted(selected[bit])) for bit in LEVELS},\n        scores={4: tuple(float(x) for x in values[4]),\n                8: tuple(float(x) for x in benefit8),\n                16: tuple(float(x) for x in benefit16)},\n        budget=budget,\n    )\n    result.verify(out_features)\n    return result\n\n\ndef allocate_model_channels(\n    layer_losses: Mapping[str, Mapping[int, torch.Tensor | Sequence[float]]],\n    budget: ThreeLevelBudget,\n    alignment: int = 1,\n    layer_minimums: Mapping[str, int] | None = None,\n    metadata: Dict[str, object] | None = None,\n) -> Dict[str, ThreeLevelAllocation]:\n    """Allocate one global bit budget across all output channels in all layers.\n\n    The returned per-layer allocations preserve original channel indices.  A\n    layer minimum is expressed as the minimum number of non-INT4 channels and\n    is satisfied before global marginal-benefit ranking.  This is the clean\n    replacement for the upstream searcher\'s per-iteration percentage split.\n    """\n    if not layer_losses:\n        raise ValueError("layer_losses must contain at least one layer")\n    minimums = dict(layer_minimums or {})\n    flattened = []\n    per_layer = {}\n    for name, losses in layer_losses.items():\n        values = {bit: torch.as_tensor(losses[bit], dtype=torch.float64).flatten()\n                  for bit in LEVELS}\n        lengths = {value.numel() for value in values.values()}\n        if len(lengths) != 1 or not lengths or next(iter(lengths)) == 0:\n            raise ValueError(f"invalid loss vectors for layer {name}")\n        n = next(iter(lengths))\n        if alignment > 1 and n < alignment:\n            raise ValueError(f"layer {name} is smaller than alignment")\n        if minimums.get(name, 0) < 0 or minimums.get(name, 0) > n:\n            raise ValueError(f"invalid layer minimum for {name}")\n        per_layer[name] = values\n        flattened.extend((float(values[4][i] - values[8][i]),\n                          float(values[8][i] - values[16][i]), name, i)\n                         for i in range(n))\n\n    total = len(flattened)\n    counts = _rounded_counts(total, budget, alignment)\n    required = sum(minimums.values())\n    if required > counts[8] + counts[16]:\n        raise ValueError("layer minimums exceed non-INT4 global budget")\n    chosen: Dict[str, Dict[int, set[int]]] = {\n        name: {4: set(), 8: set(), 16: set()} for name in per_layer\n    }\n    ranked16 = sorted(((benefit16, name, index)\n                       for _, benefit16, name, index in flattened), reverse=True)\n    for _, name, index in ranked16[:counts[16]]:\n        chosen[name][16].add(index)\n    used = {(name, index) for name in chosen for index in chosen[name][16]}\n    ranked8 = sorted(((benefit8, name, index)\n                      for benefit8, _, name, index in flattened\n                      if (name, index) not in used), reverse=True)\n    for _, name, index in ranked8[:counts[8]]:\n        chosen[name][8].add(index)\n    used.update((name, index) for name in chosen for index in chosen[name][8])\n\n    # Meet per-layer minimums without changing global precision counts.  Each\n    # swap replaces the weakest selected channel from a donor layer that stays\n    # above its own minimum with the strongest candidate from the deficient one.\n    def selected_count(layer: str) -> int:\n        return len(chosen[layer][8]) + len(chosen[layer][16])\n\n    for name in sorted(per_layer):\n        while selected_count(name) < minimums.get(name, 0):\n            candidates = []\n            for index in range(per_layer[name][4].numel()):\n                if (name, index) in used:\n                    continue\n                candidates.extend((\n                    (float(per_layer[name][4][index] - per_layer[name][8][index]),\n                     8, index),\n                    (float(per_layer[name][8][index] - per_layer[name][16][index]),\n                     16, index),\n                ))\n            swapped = False\n            for _, bit, incoming in sorted(candidates, reverse=True):\n                donors = []\n                for donor in per_layer:\n                    if selected_count(donor) <= minimums.get(donor, 0):\n                        continue\n                    for outgoing in chosen[donor][bit]:\n                        values = per_layer[donor]\n                        benefit = (values[4][outgoing] - values[8][outgoing]\n                                   if bit == 8 else\n                                   values[8][outgoing] - values[16][outgoing])\n                        donors.append((float(benefit), donor, outgoing))\n                if not donors:\n                    continue\n                _, donor, outgoing = min(donors)\n                chosen[donor][bit].remove(outgoing)\n                chosen[name][bit].add(incoming)\n                used.remove((donor, outgoing))\n                used.add((name, incoming))\n                swapped = True\n                break\n            if not swapped:\n                raise ValueError("layer minimums cannot be satisfied with exact budgets")\n    result = {}\n    for name, values in per_layer.items():\n        indices = {bit: tuple(sorted(chosen[name][bit])) for bit in LEVELS}\n        indices[4] = tuple(i for i in range(values[4].numel())\n                           if all(i not in chosen[name][bit] for bit in (8, 16)))\n        allocation = ThreeLevelAllocation(\n            indices=indices,\n            scores={4: tuple(float(x) for x in values[4]),\n                    8: tuple(float(x) for x in values[4] - values[8]),\n                    16: tuple(float(x) for x in values[8] - values[16])},\n            budget=budget,\n            metadata={**(metadata or {}), "layer_name": name,\n                      "global_channel_count": total},\n        )\n        allocation.verify(values[4].numel())\n        result[name] = allocation\n    return result\n\n\ndef allocate_model_channels_auto(\n    layer_losses: Mapping[str, Mapping[int, torch.Tensor | Sequence[float]]],\n    target_average_bits: float,\n    alignment: int = 1,\n    metadata: Dict[str, object] | None = None,\n) -> tuple[Dict[str, ThreeLevelAllocation], Dict[str, object]]:\n    """Allocate a global average-bit budget using marginal upgrade value.\n\n    Alignment applies to each layer and precision transition independently: an\n    upgrade moves ``alignment`` channels from one precision to the next within\n    a single layer. Layer tails smaller than a block stay at their precision.\n    """\n    if not layer_losses:\n        raise ValueError("layer_losses must contain at least one layer")\n    if not math.isfinite(target_average_bits) or not 4 <= target_average_bits <= 16:\n        raise ValueError("target_average_bits must be between 4 and 16")\n    if alignment < 1:\n        raise ValueError("alignment must be positive")\n\n    per_layer = {}\n    for name in sorted(layer_losses):\n        losses = layer_losses[name]\n        try:\n            values = {bit: torch.as_tensor(losses[bit], dtype=torch.float64).flatten()\n                      for bit in LEVELS}\n        except KeyError as error:\n            raise ValueError(f"missing {error.args[0]}-bit losses for layer {name}") from error\n        lengths = {value.numel() for value in values.values()}\n        if len(lengths) != 1 or next(iter(lengths), 0) == 0:\n            raise ValueError(f"invalid loss vectors for layer {name}")\n        if any(not torch.isfinite(value).all() for value in values.values()):\n            raise ValueError(f"loss vectors must be finite for layer {name}")\n        per_layer[name] = values\n\n    total = sum(values[4].numel() for values in per_layer.values())\n    requested_bit_budget = (math.floor(target_average_bits * total + 1e-9) -\n                            4 * total)\n    remaining_bits = requested_bit_budget\n    selected = {name: {4: set(range(values[4].numel())), 8: set(), 16: set()}\n                for name, values in per_layer.items()}\n\n    while True:\n        candidates = []\n        for source, target in ((4, 8), (8, 16)):\n            cost = (target - source) * alignment\n            if cost > remaining_bits:\n                continue\n            for name in per_layer:\n                ranked = [\n                    (float(per_layer[name][source][index] -\n                           per_layer[name][target][index]), index)\n                    for index in selected[name][source]\n                ]\n                ranked.sort(key=lambda item: (-item[0], item[1]))\n                if len(ranked) < alignment:\n                    continue\n                batch = ranked[:alignment]\n                benefit = sum(item[0] for item in batch)\n                candidates.append((-(benefit / cost), source, name,\n                                   batch[0][1], target, batch))\n        if not candidates:\n            break\n        _, source, name, _, target, batch = min(candidates)\n        for _, index in batch:\n            selected[name][source].remove(index)\n            selected[name][target].add(index)\n        remaining_bits -= (target - source) * alignment\n\n    counts = {bit: sum(len(selected[name][bit]) for name in selected)\n              for bit in LEVELS}\n    achieved_bits = sum(bit * counts[bit] for bit in LEVELS) / total\n    raw_percentages = {bit: 100 * counts[bit] / total for bit in LEVELS}\n    achieved_bit_budget = requested_bit_budget - remaining_bits\n    percentages = {bit: int(raw_percentages[bit]) for bit in LEVELS}\n    remainder_order = sorted(\n        LEVELS, key=lambda bit: (raw_percentages[bit] - percentages[bit], -bit),\n        reverse=True)\n    for bit in remainder_order[:100 - sum(percentages.values())]:\n        percentages[bit] += 1\n    budget = ThreeLevelBudget(percentages[4], percentages[8], percentages[16])\n\n    summary = {\n        "allocator": "marginal_benefit_bit_budget",\n        "target_average_bits": float(target_average_bits),\n        "achieved_average_bits": achieved_bits,\n        "total_channels": total,\n        "channel_counts": {str(bit): counts[bit] for bit in LEVELS},\n        "requested_bit_budget": requested_bit_budget,\n        "achieved_bit_budget": achieved_bit_budget,\n        "achieved_channel_counts": {str(bit): counts[bit] for bit in LEVELS},\n        "alignment": alignment,\n        "unused_bit_budget": remaining_bits,\n    }\n    result = {}\n    for name, values in per_layer.items():\n        allocation = ThreeLevelAllocation(\n            indices={bit: tuple(sorted(selected[name][bit])) for bit in LEVELS},\n            scores={4: tuple(float(x) for x in values[4]),\n                    8: tuple(float(x) for x in values[4] - values[8]),\n                    16: tuple(float(x) for x in values[8] - values[16])},\n            budget=budget,\n            metadata={**(metadata or {}), "layer_name": name,\n                      "global_channel_count": total,\n                      "target_average_bits": float(target_average_bits),\n                      "requested_bit_budget": requested_bit_budget,\n                      "achieved_average_bits": achieved_bits,\n                      "achieved_bit_budget": achieved_bit_budget,\n                      "achieved_channel_counts": {\n                          str(bit): counts[bit] for bit in LEVELS}},\n        )\n        allocation.verify(values[4].numel())\n        result[name] = allocation\n    return result, summary\n\n@torch.no_grad()\ndef estimate_channel_losses(activation: torch.Tensor, weight: torch.Tensor,\n                            group_size: int = 128\n                            ) -> tuple[Dict[int, torch.Tensor], torch.Tensor]:\n    """Estimate activation-aware output MSE for all three precision choices.\n\n    The same activation subset and group rules are used for every precision.\n    The returned AWQ statistic is kept separate and can be used as a tie-breaker\n    in an ablation without silently changing the marginal-loss objective.\n    """\n    if activation.shape[-1] != weight.shape[1] or weight.dim() != 2:\n        raise ValueError("activation and weight shapes are incompatible")\n    if weight.shape[1] % group_size:\n        raise ValueError("in_features must be divisible by group_size")\n    x = activation.reshape(-1, activation.shape[-1]).float()\n    reference = torch.nn.functional.linear(x, weight.float())\n    losses = {}\n    all_fp16 = ThreeLevelAllocation(\n        indices={4: (), 8: (), 16: tuple(range(weight.shape[0]))},\n        scores={bit: (0.0,) * weight.shape[0] for bit in LEVELS},\n        budget=ThreeLevelBudget(0, 0, 100),\n    )\n    for bit in LEVELS:\n        indices = {level: () for level in LEVELS}\n        indices[bit] = tuple(range(weight.shape[0]))\n        allocation = ThreeLevelAllocation(\n            indices=indices, scores=all_fp16.scores,\n            budget=ThreeLevelBudget(100 if bit == 4 else 0,\n                                    100 if bit == 8 else 0,\n                                    100 if bit == 16 else 0),\n        )\n        output = fake_linear(x, weight.float(), allocation, group_size)\n        losses[bit] = (output - reference).pow(2).mean(dim=0).cpu()\n    awq_stat = x.abs().mean(dim=0).cpu()\n    return losses, awq_stat\n\n\ndef fake_linear(x: torch.Tensor, weight: torch.Tensor,\n                allocation: ThreeLevelAllocation,\n                group_size: int = 128) -> torch.Tensor:\n    """Reference output for a three-level weight-only linear layer."""\n    if weight.dim() != 2 or x.shape[-1] != weight.shape[1]:\n        raise ValueError("incompatible x/weight shapes")\n    output = torch.empty((*x.shape[:-1], weight.shape[0]), dtype=x.dtype, device=x.device)\n    for bit in LEVELS:\n        indices = allocation.indices[bit]\n        if not indices:\n            continue\n        chunk = weight[list(indices)]\n        if bit == 16:\n            quantized = chunk\n        else:\n            if chunk.shape[1] % group_size:\n                raise ValueError("in_features must be divisible by group_size")\n            groups = chunk.reshape(chunk.shape[0], -1, group_size)\n            if bit == 8:\n                maximum = groups.abs().amax(dim=-1, keepdim=True).clamp_min(1e-5)\n                quantized = (groups / (maximum / 127)).round().clamp(-128, 127) * (maximum / 127)\n            else:\n                minimum = groups.amin(dim=-1, keepdim=True)\n                maximum = groups.amax(dim=-1, keepdim=True)\n                scale = (maximum - minimum).clamp_min(1e-5) / 15\n                zero = (-minimum / scale).round().clamp(0, 15)\n                quantized = ((groups / scale).round() + zero).clamp(0, 15)\n                quantized = (quantized - zero) * scale\n            quantized = quantized.reshape_as(chunk)\n        output[..., list(indices)] = torch.nn.functional.linear(x, quantized)\n    return output', 'mixllm/nn/modules/mixllm_config.py': '# Copyright (c) Microsoft Corporation.\n# SPDX-License-Identifier: MIT\n\nimport os\nimport json\nfrom typing import Dict, Optional, List, Tuple\nfrom dataclasses import dataclass, field\nfrom transformers.utils.hub import PushToHubMixin\n\n\n@dataclass\nclass MixLLMConfig(PushToHubMixin):\n    quant_method: str = field(default="mixllm")\n    ratio: float = field(default=1.0)\n    # Legacy MixLLM uses ratio as the INT8 fraction.  These fields are\n    # optional so existing two-level checkpoints remain loadable.\n    precision_percentages: Optional[Dict[str, int]] = None\n    allocation_file: Optional[str] = None\n    allocation_version: int = 1\n    group_size: int = 128\n    backend: str = "auto"\n    format_version: int = 2\n    model_revision: Optional[str] = None\n    calibration_seed: Optional[int] = None\n    dtype: str = "float16"\n    kernel_capability: Optional[Tuple[int, int]] = None\n    config_file_name = "config.json"\n    modules_to_not_convert: Optional[List] = None\n\n    @classmethod\n    def from_dict(cls, quant_config: Dict = {}):\n        if not quant_config:\n            quant_config = cls()\n        else:\n            supported = {\n                key: value for key, value in quant_config.items()\n                if key in cls.__dataclass_fields__\n            }\n            quant_config = cls(**supported)\n\n        if quant_config.precision_percentages is None and quant_config.ratio != 1.0:\n            int8 = round(float(quant_config.ratio) * 100)\n            quant_config.precision_percentages = {"4": 100 - int8, "8": int8, "16": 0}\n        if quant_config.precision_percentages is not None:\n            percentages = quant_config.precision_percentages\n            normalized = {str(bit): int(percentages.get(str(bit), 0))\n                          for bit in (4, 8, 16)}\n            if any(value < 0 for value in normalized.values()) or sum(normalized.values()) != 100:\n                raise ValueError("precision_percentages for 4/8/16 must sum to 100")\n            quant_config.precision_percentages = normalized\n        if quant_config.group_size <= 0:\n            raise ValueError("group_size must be positive")\n        if quant_config.format_version not in (1, 2):\n            raise ValueError("unsupported MixLLM config format_version")\n        if quant_config.dtype not in {"float16", "bfloat16"}:\n            raise ValueError("dtype must be float16 or bfloat16")\n        if quant_config.kernel_capability is not None:\n            capability = tuple(int(value) for value in quant_config.kernel_capability)\n            if len(capability) != 2 or any(value < 0 for value in capability):\n                raise ValueError("kernel_capability must be a CUDA (major, minor) pair")\n            quant_config.kernel_capability = capability\n\n        return quant_config\n\n    def to_dict(self):\n        return {\n            "ratio": self.ratio,\n            "modules_to_not_convert": self.modules_to_not_convert,\n            "precision_percentages": self.precision_percentages,\n            "allocation_file": self.allocation_file,\n            "allocation_version": self.allocation_version,\n            "group_size": self.group_size,\n            "backend": self.backend,\n            "format_version": self.format_version,\n            "model_revision": self.model_revision,\n            "calibration_seed": self.calibration_seed,\n            "dtype": self.dtype,\n            "kernel_capability": self.kernel_capability,\n        }\n\n    def to_transformers_dict(self):\n        return {\n            "quant_method": self.quant_method,\n            "ratio": self.ratio,\n            "modules_to_not_convert": self.modules_to_not_convert,\n            "precision_percentages": self.precision_percentages,\n            "allocation_file": self.allocation_file,\n            "allocation_version": self.allocation_version,\n            "group_size": self.group_size,\n            "backend": self.backend,\n            "format_version": self.format_version,\n            "model_revision": self.model_revision,\n            "calibration_seed": self.calibration_seed,\n            "dtype": self.dtype,\n            "kernel_capability": self.kernel_capability,\n        }\n\n    def from_transformers_dict(self, transformers_dict: Dict):\n        return {\n            "quant_method":\n                transformers_dict.get("quant_method"),\n            "ratio":\n                transformers_dict.get("ratio"),\n            "modules_to_not_convert":\n                transformers_dict.get("modules_to_not_convert"),\n            "precision_percentages":\n                transformers_dict.get("precision_percentages"),\n            "allocation_file": transformers_dict.get("allocation_file"),\n            "allocation_version": transformers_dict.get("allocation_version", 1),\n            "group_size": transformers_dict.get("group_size", 128),\n            "backend": transformers_dict.get("backend", "auto"),\n            "format_version": transformers_dict.get("format_version", 1),\n            "model_revision": transformers_dict.get("model_revision"),\n            "calibration_seed": transformers_dict.get("calibration_seed"),\n            "dtype": transformers_dict.get("dtype", "float16"),\n            "kernel_capability": transformers_dict.get("kernel_capability"),\n        }\n', 'mixllm/nn/modules/three_level_linear.py': '"""Packed FP16/INT8/INT4 output-feature linear for MixLLM."""\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\nimport torch\nfrom torch import nn\n\nfrom mixllm.quantization.three_level import LEVELS, ThreeLevelAllocation\n\n\ndef _pack_uint4(codes: torch.Tensor) -> torch.Tensor:\n    if codes.shape[-1] % 2:\n        raise ValueError("INT4 input width must be even")\n    values = codes.to(torch.uint8)\n    return (values[..., 0::2] | (values[..., 1::2] << 4)).contiguous()\n\n\ndef _unpack_uint4(packed: torch.Tensor) -> torch.Tensor:\n    output = torch.empty((*packed.shape[:-1], packed.shape[-1] * 2),\n                         dtype=torch.uint8, device=packed.device)\n    output[..., 0::2] = packed & 0x0F\n    output[..., 1::2] = packed >> 4\n    return output\n\n\nclass ThreeLevelLinear(nn.Module):\n    """Reference backend and serialization contract for a three-level op."""\n\n    quant_method = "mixllm_three_level"\n\n    def __init__(self, in_features: int, out_features: int, group_size: int = 128,\n                 bias: Optional[torch.Tensor] = None) -> None:\n        super().__init__()\n        if in_features <= 0 or out_features <= 0 or group_size <= 0:\n            raise ValueError("linear dimensions and group_size must be positive")\n        if in_features % group_size:\n            raise ValueError("in_features must be divisible by group_size")\n        self.in_features = in_features\n        self.out_features = out_features\n        self.group_size = group_size\n        self.register_buffer("weight_fp16", torch.empty(0, in_features, dtype=torch.float16))\n        self.register_buffer("weight_int8", torch.empty(0, in_features, dtype=torch.int8))\n        self.register_buffer("scale_int8", torch.empty(0, in_features // group_size, dtype=torch.float16))\n        self.register_buffer("weight_int4", torch.empty(0, in_features // 2, dtype=torch.uint8))\n        self.register_buffer("scale_int4", torch.empty(0, in_features // group_size, dtype=torch.float16))\n        self.register_buffer("zero_int4", torch.empty(0, in_features // group_size, dtype=torch.uint8))\n        for bit in LEVELS:\n            self.register_buffer(f"indices_{bit}", torch.empty(0, dtype=torch.int32))\n        self.register_buffer("bias", None if bias is None else bias.detach().to(torch.float16))\n        # SM75 runtime caches are not buffers because they contain no model state.\n        self._sm75_fp16_placeholders = None\n        self._sm75_int4_expanded = None\n\n    def _apply(self, fn, recurse=True):\n        self._sm75_fp16_placeholders = None\n        self._sm75_int4_expanded = None\n        return super()._apply(fn, recurse)\n\n    @classmethod\n    @torch.no_grad()\n    def from_weight(cls, weight: torch.Tensor, allocation: ThreeLevelAllocation,\n                    group_size: int = 128,\n                    bias: Optional[torch.Tensor] = None) -> "ThreeLevelLinear":\n        if weight.dim() != 2:\n            raise ValueError("weight must have shape [out_features, in_features]")\n        out_features, in_features = weight.shape\n        allocation.verify(out_features)\n        layer = cls(in_features, out_features, group_size, bias).to(weight.device)\n        groups = in_features // group_size\n\n        for bit in LEVELS:\n            indices = torch.tensor(allocation.indices[bit], dtype=torch.int32,\n                                   device=weight.device)\n            setattr(layer, f"indices_{bit}", indices)\n            if not indices.numel():\n                continue\n            chunk = weight.index_select(0, indices.long()).float()\n            if bit == 16:\n                layer.weight_fp16 = chunk.to(torch.float16).contiguous()\n            elif bit == 8:\n                viewed = chunk.reshape(-1, groups, group_size)\n                scale = (viewed.abs().amax(-1) / 127).clamp_min(1e-5)\n                codes = (viewed / scale.unsqueeze(-1)).round().clamp(-128, 127)\n                layer.weight_int8 = codes.to(torch.int8).reshape(-1, in_features).contiguous()\n                layer.scale_int8 = scale.to(torch.float16).contiguous()\n            else:\n                viewed = chunk.reshape(-1, groups, group_size)\n                minimum, maximum = viewed.amin(-1), viewed.amax(-1)\n                scale = ((maximum - minimum) / 15).clamp_min(1e-5)\n                zero = (-minimum / scale).round().clamp(0, 15)\n                codes = ((viewed / scale.unsqueeze(-1)).round() + zero.unsqueeze(-1)).clamp(0, 15)\n                layer.weight_int4 = _pack_uint4(codes.to(torch.uint8).reshape(-1, in_features))\n                layer.scale_int4 = scale.to(torch.float16).contiguous()\n                layer.zero_int4 = zero.to(torch.uint8).contiguous()\n        return layer\n\n    @torch.no_grad()\n    def dequantize_weight(self) -> torch.Tensor:\n        device = self.weight_fp16.device\n        output = torch.empty(self.out_features, self.in_features,\n                             dtype=torch.float32, device=device)\n        if self.indices_16.numel():\n            output.index_copy_(0, self.indices_16.long(), self.weight_fp16.float())\n        if self.indices_8.numel():\n            scales = self.scale_int8.float().repeat_interleave(self.group_size, dim=1)\n            output.index_copy_(0, self.indices_8.long(), self.weight_int8.float() * scales)\n        if self.indices_4.numel():\n            codes = _unpack_uint4(self.weight_int4).float()\n            scales = self.scale_int4.float().repeat_interleave(self.group_size, dim=1)\n            zeros = self.zero_int4.float().repeat_interleave(self.group_size, dim=1)\n            output.index_copy_(0, self.indices_4.long(), (codes - zeros) * scales)\n        return output\n\n    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,\n                              missing_keys, unexpected_keys, error_msgs):\n        self._sm75_fp16_placeholders = None\n        self._sm75_int4_expanded = None\n        # Two-level checkpoints predate the FP16 partition. Treat its omitted\n        # tensors as an empty partition while preserving strict loading for all\n        # other packed state.\n        legacy_defaults = {\n            "weight_fp16": self.weight_fp16,\n            "indices_16": self.indices_16,\n        }\n        for name, value in legacy_defaults.items():\n            state_dict.setdefault(prefix + name, value)\n        variable_buffers = (\n            "weight_fp16", "weight_int8", "scale_int8", "weight_int4",\n            "scale_int4", "zero_int4", "indices_4", "indices_8", "indices_16",\n        )\n        for name in variable_buffers:\n            key = prefix + name\n            if key in state_dict:\n                setattr(self, name, torch.empty_like(state_dict[key], device=self.weight_fp16.device))\n        super()._load_from_state_dict(state_dict, prefix, local_metadata, strict,\n                                      missing_keys, unexpected_keys, error_msgs)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        if x.shape[-1] != self.in_features:\n            raise ValueError(f"expected input width {self.in_features}, got {x.shape[-1]}")\n        result = torch.nn.functional.linear(x.float(), self.dequantize_weight(),\n                                            None if self.bias is None else self.bias.float())\n        return result.to(x.dtype)\n', 'mixllm/nn/modules/ops.py': '# Copyright (c) Microsoft Corporation.\n# SPDX-License-Identifier: MIT\n\nimport torch\n\n__all__ = ["quantize", "transpose", "mixllm_gemm", "mixllm_three_level_gemm"]\n\n\ndef transpose(a):\n    return torch.ops.kernels_mixllm.transpose(a)\n\n\ndef quantize(a):\n    return torch.ops.kernels_mixllm.quantize(a)\n\n\ndef mixllm_gemm(a, scale_act, zero, scale_int8, scale_int4, indices_int8,\n                indices_int4, b_int8, b_int4):\n    return torch.ops.kernels_mixllm.gemm(a, scale_act, zero, scale_int8,\n                                         scale_int4, indices_int8, indices_int4,\n                                         b_int8, b_int4)\n\n\ndef mixllm_three_level_gemm(a, scale_act, zero_int4, scale_int8,\n                            scale_int4, indices_int8, indices_int4,\n                            indices_fp16, b_int8, b_int4, b_fp16):\n    """Correctness-first 4/8/16 ABI using the existing MixLLM quantized op.\n\n    The quantized partitions run through the upstream CUDA extension. The FP16\n    partition is computed with PyTorch GEMM, and all outputs are scattered to\n    their original output-feature positions. A future fused op can replace this\n    implementation without changing the checkpoint contract.\n    """\n    partitions = (indices_int4, indices_int8, indices_fp16)\n    total_n = sum(indices.numel() for indices in partitions)\n    if total_n == 0:\n        raise ValueError("at least one output partition is required")\n    complete = torch.cat(partitions).long()\n    if complete.unique().numel() != total_n or complete.min().item() != 0 or complete.max().item() != total_n - 1:\n        raise ValueError("4/8/16 indices must form a complete output-channel partition")\n\n    m, k = a.shape\n    if k % 128:\n        raise ValueError("current MixLLM CUDA ABI requires K divisible by 128")\n    output = torch.empty((m, total_n), dtype=torch.float16, device=a.device)\n    n8, n4 = indices_int8.numel(), indices_int4.numel()\n    if n8 + n4:\n        local_int8 = torch.arange(n8, dtype=torch.int32, device=a.device)\n        local_int4 = torch.arange(n8, n8 + n4, dtype=torch.int32, device=a.device)\n        quantized = mixllm_gemm(\n            a, scale_act, zero_int4, scale_int8, scale_int4,\n            local_int8, local_int4, b_int8, b_int4,\n        )\n        if n8 and n4:\n            quantized = quantized.t().contiguous()\n        output.index_copy_(1, torch.cat((indices_int8, indices_int4)).long(), quantized)\n\n    if indices_fp16.numel():\n        groups = k // 128\n        activation = (\n            a.float().reshape(m, groups, 128)\n            * scale_act[:, :m].t().reshape(m, groups, 1).float()\n        ).reshape(m, k)\n        fp16_result = activation @ b_fp16.float().t()\n        output.index_copy_(1, indices_fp16.long(), fp16_result.to(torch.float16))\n    return output\n\n\n@torch.library.register_fake("kernels_mixllm::quantize")\ndef quantize_abstract(a):\n    torch._check(a.dim() == 2, "Input must be a 2D tensor")\n    m = a.shape[0]\n    n = a.shape[1]\n    group_size = 128\n    torch._check(a.is_cuda, "Input must be on CUDA device")\n    torch._check(a.dtype == torch.float16, "Input must be float16")\n    torch._check(\n        n % group_size == 0,\n        "Input must have a second dimension that is a multiple of group_size")\n\n    m_round_even = m + (m % 2)\n    return (torch.empty((m, n), dtype=torch.int8, device="cuda:0"),\n            torch.empty((n // group_size, m_round_even),\n                        dtype=torch.float16,\n                        device="cuda:0"))\n\n\n@torch.library.register_fake("kernels_mixllm::transpose")\ndef transpose_abstract(a):\n    torch._check(a.dim() == 2, "Input must be a 2D tensor")\n    m = a.shape[0]\n    n = a.shape[1]\n    torch._check(a.is_cuda, "Input must be on CUDA device")\n    torch._check(a.dtype == torch.float16, "Input must be float16")\n    return torch.empty((n, m), dtype=torch.float16, device="cuda:0")\n\n\n@torch.library.register_fake("kernels_mixllm::gemm")\ndef mixllm_gemm_abstract(a, scale_act, zero, scale_int8, scale_int4,\n                         indices_int8, indices_int4, b_int8, b_int4):\n    torch._check(a.is_cuda, "Input tensor A must be on CUDA device")\n    torch._check(a.dtype == torch.int8, "Input tensor A must be int8")\n    torch._check(scale_act.dtype == torch.float16,\n                 "Scale activation tensor must be float16")\n    torch._check(zero.dtype == torch.uint8, "Zero tensor must be uint8")\n    torch._check(scale_int8.dtype == torch.float16,\n                 "Scale int8 tensor must be float16")\n    torch._check(scale_int4.dtype == torch.float16,\n                 "Scale int4 tensor must be float16")\n    torch._check(indices_int8.dtype == torch.int32,\n                 "Indices int8 tensor must be int32")\n    torch._check(indices_int4.dtype == torch.int32,\n                 "Indices int4 tensor must be int32")\n    torch._check(b_int8.dtype == torch.int8, "B int8 tensor must be int8")\n    torch._check(b_int4.dtype == torch.uint8, "B int4 tensor must be uint8")\n    torch._check(b_int8.is_cuda, "B int8 tensor must be on CUDA device")\n    torch._check(b_int4.is_cuda, "B int4 tensor must be on CUDA device")\n    torch._check(a.dim() == 2, "Input tensor A must be 2D")\n    torch._check(scale_act.dim() == 2, "Scale activation tensor must be 2D")\n    torch._check(zero.dim() == 2, "Zero tensor must be 2D")\n    torch._check(scale_int8.dim() == 2, "Scale int8 tensor must be 2D")\n    torch._check(scale_int4.dim() == 2, "Scale int4 tensor must be 2D")\n    torch._check(indices_int8.dim() == 1, "Indices int8 tensor must be 1D")\n    torch._check(indices_int4.dim() == 1, "Indices int4 tensor must be 1D")\n    torch._check(b_int8.dim() == 2, "B int8 tensor must be 2D")\n    torch._check(b_int4.dim() == 2, "B int4 tensor must be 2D")\n\n    m = a.shape[0]\n    n = (indices_int4.numel() + indices_int8.numel())\n    is_row_major = (indices_int4.numel() == 0 or indices_int8.numel() == 0)\n    if is_row_major:\n        c = torch.empty((m, n), dtype=torch.float16, device="cuda:0")\n    else:\n        c = torch.empty((n, m), dtype=torch.float16, device="cuda:0")\n\n    return c\n', 'mixllm/runtime_capability.py': '"""Runtime capability gates for the MixLLM CUDA backends."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Optional\n\n\n@dataclass(frozen=True)\nclass RuntimeCapability:\n    major: int\n    minor: int\n    requested_backend: str = "auto"\n    sm75_available: bool = False\n\n    @property\n    def compute_capability(self) -> float:\n        return self.major + self.minor / 10\n\n    @property\n    def supports_ampere_mixllm(self) -> bool:\n        return self.major >= 8\n\n    @property\n    def supports_sm75_backend(self) -> bool:\n        return self.sm75_available and (self.major, self.minor) == (7, 5)\n\n    def select_backend(self) -> str:\n        if self.requested_backend not in {"auto", "sm75", "ampere", "reference"}:\n            raise ValueError(f"unknown MixLLM backend: {self.requested_backend}")\n        if self.requested_backend == "reference":\n            return "reference"\n        if self.requested_backend == "ampere":\n            if not self.supports_ampere_mixllm:\n                raise RuntimeError("Ampere MixLLM backend requires compute capability >= 8.0")\n            return "ampere"\n        if self.requested_backend == "sm75":\n            if not self.supports_sm75_backend:\n                raise RuntimeError(\n                    "SM75 backend requires compute capability 7.5 and a validated SM75 build"\n                )\n            return "sm75"\n        if self.supports_ampere_mixllm:\n            return "ampere"\n        if self.supports_sm75_backend:\n            return "sm75"\n        return "reference"\n\n\ndef detect_runtime(torch_module) -> Optional[RuntimeCapability]:\n    """Detect CUDA capability without importing torch at module import time."""\n    if not torch_module.cuda.is_available():\n        return None\n    major, minor = torch_module.cuda.get_device_capability()\n    return RuntimeCapability(major, minor)', 'mixllm/sm75_backend.py': '"""Build and invoke the single-launch SM75 three-level Tensor Core backend."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom statistics import median\nfrom typing import Dict, Iterable, Optional\n\n\n_LOADED = False\n\n\ndef load_sm75_backend(torch_module, build_directory: Optional[str | Path] = None) -> None:\n    """JIT-build the SM75-only torch operator and load it into this process."""\n    global _LOADED\n    if _LOADED:\n        return\n    if not torch_module.cuda.is_available():\n        raise RuntimeError("SM75 backend requires CUDA")\n    capability = tuple(torch_module.cuda.get_device_capability())\n    if capability != (7, 5):\n        raise RuntimeError(f"SM75 backend requires capability 7.5, got {capability}")\n\n    from torch.utils.cpp_extension import load\n\n    source = Path(__file__).resolve().parent / "kernels" / "three_level_sm75.cu"\n    kwargs = {}\n    if build_directory is not None:\n        directory = Path(build_directory)\n        directory.mkdir(parents=True, exist_ok=True)\n        kwargs["build_directory"] = str(directory)\n    load(\n        name="mixllm_sm75_backend",\n        sources=[str(source)],\n        extra_cuda_cflags=["-O3", "-lineinfo", "-gencode=arch=compute_75,code=sm_75"],\n        extra_cflags=["-O3"],\n        is_python_module=False,\n        verbose=True,\n        **kwargs,\n    )\n    _LOADED = True\n\n\ndef quantize_activation(x, torch_module, group_size: int = 128):\n    """Reference symmetric per-row, per-group INT8 activation quantization."""\n    if x.dim() != 2 or x.shape[1] % group_size:\n        raise ValueError("activation must be 2D with width divisible by group_size")\n    rows, width = x.shape\n    if rows == 0:\n        return (\n            torch_module.empty_like(x, dtype=torch_module.int8),\n            torch_module.empty(\n                width // group_size, 0, device=x.device,\n                dtype=torch_module.float16,\n            ),\n        )\n    grouped = x.float().reshape(rows, width // group_size, group_size)\n    scales = (grouped.abs().amax(dim=-1) / 127.0).clamp_min(1e-8)\n    quantized = (grouped / scales.unsqueeze(-1)).round().clamp(-127, 127)\n    return (quantized.to(torch_module.int8).reshape(rows, width),\n            scales.t().to(torch_module.float16).contiguous())\n\n\ndef quantize_activation_native(x, torch_module, group_size: int = 128):\n    """Run the native SM75 activation quantizer used by the inference path."""\n    if not _LOADED:\n        raise RuntimeError("call load_sm75_backend before using the SM75 operator")\n    if x.dim() != 2 or x.shape[1] % group_size:\n        raise ValueError("activation must be 2D with width divisible by group_size")\n    if group_size != 128 or x.dtype != torch_module.float16:\n        raise ValueError("native SM75 quantization requires FP16 and group_size=128")\n    if x.shape[0] == 0:\n        return (\n            torch_module.empty_like(x, dtype=torch_module.int8),\n            torch_module.empty(\n                x.shape[1] // group_size, 0, device=x.device,\n                dtype=torch_module.float16,\n            ),\n        )\n    return torch_module.ops.mixllm_sm75.quantize_activation(x.contiguous())\n\n\ndef quantized_reference_prequantized(\n    module, x, quantized, scales, torch_module,\n):\n    """Reference the GEMM using a caller-provided activation quantization."""\n    rows, width = x.shape\n    reconstructed = (\n        quantized.float().reshape(rows, -1, module.group_size)\n        * scales.t().float().unsqueeze(-1)\n    ).reshape(rows, width)\n    output = torch_module.empty(\n        rows, module.out_features, device=x.device, dtype=torch_module.float32,\n    )\n    dense = module.dequantize_weight()\n    for bit, activation in ((4, reconstructed), (8, reconstructed), (16, x)):\n        indices = getattr(module, f"indices_{bit}")\n        if not indices.numel():\n            continue\n        weight = dense.index_select(0, indices.long())\n        values = torch_module.nn.functional.linear(\n            activation if bit != 16 else activation.to(torch_module.float16),\n            weight if bit != 16 else weight.to(torch_module.float16),\n        ).float()\n        output.index_copy_(1, indices.long(), values)\n    return output\n\n\ndef quantized_reference(module, x, torch_module):\n    """Reference the complete mixed activation contract used by the SM75 op."""\n    quantized, scales = quantize_activation(x, torch_module, module.group_size)\n    return quantized_reference_prequantized(\n        module, x, quantized, scales, torch_module,\n    )\n\n\ndef _validate_partition(module, x, torch_module):\n    signature = (\n        int(module.out_features),\n        *((id(indices), int(indices._version), int(indices.numel()), indices.device)\n          for indices in (module.indices_4, module.indices_8, module.indices_16)),\n    )\n    if getattr(module, "_sm75_partition_validation", None) == signature:\n        return\n    if x.is_cuda and torch_module.cuda.is_current_stream_capturing():\n        raise RuntimeError("SM75 partition must be validated before CUDA graph capture")\n    indices = (module.indices_4, module.indices_8, module.indices_16)\n    complete = torch_module.cat(indices).long()\n    expected = torch_module.arange(module.out_features, device=x.device)\n    if complete.numel() != module.out_features or not torch_module.equal(\n            complete.sort().values, expected):\n        raise ValueError(\n            "4/8/16 indices must form a complete output partition without "\n            "duplicates, missing channels, or out-of-range indices"\n        )\n    module._sm75_partition_validation = signature\n\n\ndef _expanded_int4_for_prefill(module, x, torch_module):\n    """Expand packed INT4 once per packed state/device for the SM75 prefill path."""\n    if x.shape[0] == 1 or not module.indices_4.numel():\n        # Decode does not read this argument. Empty INT4 prefill still needs the\n        # ABI-compatible [0, K] shape without allocating a separate tensor.\n        return module.weight_int8[:0]\n    signature = (\n        module.weight_int4.device,\n        id(module.weight_int4), int(module.weight_int4._version),\n        id(module.zero_int4), int(module.zero_int4._version),\n        tuple(module.weight_int4.shape), tuple(module.zero_int4.shape),\n    )\n    cached = module._sm75_int4_expanded\n    if cached is not None and cached[0] == signature:\n        return cached[1]\n    if x.is_cuda and torch_module.cuda.is_current_stream_capturing():\n        raise RuntimeError("SM75 INT4 prefill cache must be initialized before CUDA graph capture")\n    packed = module.weight_int4\n    expanded = torch_module.empty(\n        packed.shape[0], packed.shape[1] * 2,\n        dtype=torch_module.uint8, device=packed.device,\n    )\n    expanded[:, 0::2] = packed & 0x0f\n    expanded[:, 1::2] = packed >> 4\n    zeros = module.zero_int4.repeat_interleave(\n        module.group_size, dim=1,\n    ).to(torch_module.int16)\n    expanded = (expanded.to(torch_module.int16) - zeros).to(\n        torch_module.int8,\n    ).contiguous()\n    module._sm75_int4_expanded = (signature, expanded)\n    return expanded\n\n\ndef three_level_linear_prequantized(module, x, input_int8, scale_act, torch_module):\n    """Run the physical GEMM kernel with precomputed activation quantization."""\n    if not _LOADED:\n        raise RuntimeError("call load_sm75_backend before using the SM75 operator")\n    if x.dim() != 2:\n        raise ValueError("SM75 correctness backend currently requires a 2D input")\n    packed_tensors = (\n        module.weight_int4, module.scale_int4, module.zero_int4, module.indices_4,\n        module.weight_int8, module.scale_int8, module.indices_8,\n        module.weight_fp16, module.indices_16,\n    )\n    tensors = (input_int8, scale_act, *packed_tensors)\n    if any(tensor.device != x.device for tensor in tensors):\n        raise ValueError("input and all operator tensors must be on the same device")\n    if x.dtype != torch_module.float16:\n        raise ValueError("SM75 Tensor Core backend requires float16 activation input")\n    if module.group_size != 128:\n        raise ValueError("SM75 Tensor Core backend currently requires group_size=128")\n    _validate_partition(module, x, torch_module)\n    if x.shape[0] == 0:\n        return torch_module.empty(\n            0, module.out_features, device=x.device, dtype=torch_module.float32,\n        )\n    expanded_int4 = _expanded_int4_for_prefill(module, x, torch_module)\n    return torch_module.ops.mixllm_sm75._three_level_linear_v2_unchecked(\n        x if x.is_contiguous() else x.contiguous(),\n        input_int8 if input_int8.is_contiguous() else input_int8.contiguous(),\n        scale_act if scale_act.is_contiguous() else scale_act.contiguous(),\n        module.weight_int4,\n        expanded_int4,\n        *(tensor if tensor.is_contiguous() else tensor.contiguous()\n          for tensor in packed_tensors[1:]),\n    )\n\n\ndef _fp16_abi_placeholders(module, x, torch_module):\n    """Return reusable tensors required but unread by the pure-FP16 kernel path."""\n    key = (x.device, tuple(x.shape))\n    cached = module._sm75_fp16_placeholders\n    if cached is None or cached[0] != key:\n        rows, width = x.shape\n        cached = (\n            key,\n            torch_module.empty_like(x, dtype=torch_module.int8),\n            torch_module.empty(\n                width // module.group_size, rows, device=x.device,\n                dtype=torch_module.float16,\n            ),\n        )\n        module._sm75_fp16_placeholders = cached\n    return cached[1], cached[2]\n\n\ndef three_level_linear(module, x, torch_module):\n    """Dispatch by shape/partition and run the single three-level GEMM ABI."""\n    if not _LOADED:\n        raise RuntimeError("call load_sm75_backend before using the SM75 operator")\n    if x.dim() != 2:\n        raise ValueError("SM75 correctness backend currently requires a 2D input")\n    packed_tensors = (\n        module.weight_int4, module.scale_int4, module.zero_int4, module.indices_4,\n        module.weight_int8, module.scale_int8, module.indices_8,\n        module.weight_fp16, module.indices_16,\n    )\n    if any(tensor.device != x.device for tensor in packed_tensors):\n        raise ValueError("input and all packed tensors must be on the same device")\n    _validate_partition(module, x, torch_module)\n    if x.shape[0] == 0:\n        return torch_module.empty(\n            0, module.out_features, device=x.device, dtype=torch_module.float32,\n        )\n    if not module.indices_4.numel() and not module.indices_8.numel():\n        input_int8, scale_act = _fp16_abi_placeholders(module, x, torch_module)\n        return three_level_linear_prequantized(\n            module, x, input_int8, scale_act, torch_module,\n        )\n    input_int8, scale_act = quantize_activation_native(\n        x, torch_module, module.group_size,\n    )\n    return three_level_linear_prequantized(\n        module, x, input_int8, scale_act, torch_module,\n    )\n\n\ndef benchmark_sm75_backend(\n    module,\n    rows: Iterable[int],\n    torch_module,\n    warmup: int = 10,\n    iterations: int = 50,\n) -> Dict[str, object]:\n    """Measure the SM75 operator against its dequantized dense reference.\n\n    Timings use per-iteration CUDA events and synchronize only after all events\n    have been recorded. This is a kernel-level benchmark, not tokens/second.\n    """\n    if warmup < 1 or iterations < 2:\n        raise ValueError("benchmark requires warmup >= 1 and iterations >= 2")\n    if not _LOADED:\n        raise RuntimeError("call load_sm75_backend before benchmarking")\n    dense_weight = module.dequantize_weight().to(torch_module.float16)\n    results = []\n    partition_counts = {\n        bit: int(getattr(module, f"indices_{bit}").numel())\n        for bit in (4, 8, 16)\n    }\n    total_channels = sum(partition_counts.values())\n    average_weight_bits = (\n        sum(bit * partition_counts[bit] for bit in (4, 8, 16))\n        / total_channels\n    )\n\n    def measure(callable_):\n        for _ in range(warmup):\n            callable_()\n        torch_module.cuda.synchronize()\n        pairs = []\n        for _ in range(iterations):\n            start = torch_module.cuda.Event(enable_timing=True)\n            end = torch_module.cuda.Event(enable_timing=True)\n            start.record()\n            callable_()\n            end.record()\n            pairs.append((start, end))\n        torch_module.cuda.synchronize()\n        values = sorted(float(start.elapsed_time(end)) for start, end in pairs)\n        p95_index = min(len(values) - 1, int(0.95 * len(values)))\n        return {"p50_ms": median(values), "p95_ms": values[p95_index]}\n\n    for row_count in rows:\n        if int(row_count) <= 0:\n            raise ValueError("benchmark row counts must be positive")\n        x = torch_module.randn(\n            int(row_count), module.in_features, device=module.weight_fp16.device,\n            dtype=torch_module.float16,\n        )\n        input_int8, scale_act = quantize_activation_native(\n            x, torch_module, module.group_size,\n        )\n        actual = three_level_linear_prequantized(\n            module, x, input_int8, scale_act, torch_module,\n        )\n        operator_reference = quantized_reference_prequantized(\n            module, x, input_int8, scale_act, torch_module,\n        )\n        dense_reference = torch_module.nn.functional.linear(x, dense_weight).float()\n        max_error = float((actual - operator_reference).abs().max().item())\n        dense_semantic_error = float((actual - dense_reference).abs().max().item())\n        quantization = measure(\n            lambda: quantize_activation_native(x, torch_module, module.group_size),\n        )\n        gemm = measure(lambda: three_level_linear_prequantized(\n            module, x, input_int8, scale_act, torch_module,\n        ))\n        end_to_end = measure(lambda: three_level_linear(module, x, torch_module))\n        dense = measure(lambda: torch_module.nn.functional.linear(x, dense_weight))\n        results.append({\n            "rows": int(row_count),\n            "activation_quantization": quantization,\n            "sm75_gemm": gemm,\n            "sm75_end_to_end": end_to_end,\n            "dense_fp16": dense,\n            "gemm_p50_ratio_vs_dense": (\n                gemm["p50_ms"] / max(dense["p50_ms"], 1e-9)\n            ),\n            "gemm_p50_speedup_vs_dense": (\n                dense["p50_ms"] / max(gemm["p50_ms"], 1e-9)\n            ),\n            "end_to_end_p50_ratio_vs_dense": (\n                end_to_end["p50_ms"] / max(dense["p50_ms"], 1e-9)\n            ),\n            "end_to_end_p50_speedup_vs_dense": (\n                dense["p50_ms"] / max(end_to_end["p50_ms"], 1e-9)\n            ),\n            "max_abs_error": max_error,\n            "max_abs_error_vs_dense_fp16": dense_semantic_error,\n        })\n    return {\n        "status": "measured",\n        "in_features": int(module.in_features),\n        "out_features": int(module.out_features),\n        "partition_counts": partition_counts,\n        "average_weight_bits": average_weight_bits,\n        "weight_bandwidth_upper_bound_vs_fp16": 16.0 / average_weight_bits,\n        "shapes": results,\n    }\n', 'mixllm/model_gate.py': '"""Small-to-large model validation for the three-level MixLLM format."""\n\nfrom __future__ import annotations\n\nfrom collections import OrderedDict\nimport math\nimport time\nfrom typing import Dict, Iterable, Mapping\n\nimport torch\nfrom torch import nn\n\nfrom mixllm.nn.modules.three_level_linear import ThreeLevelLinear\nfrom mixllm.quantization.three_level import (\n    ThreeLevelBudget,\n    allocate_model_channels,\n    allocate_model_channels_auto,\n    estimate_channel_losses,\n)\n\n\nDEFAULT_TEXTS = (\n    "Mixed precision protects channels whose quantization error matters most.",\n    "A reproducible benchmark separates numerical quality from kernel speed.",\n    "The quick model gate must pass before running the seven billion parameter model.",\n    "CUDA events measure device work after warmup and explicit synchronization.",\n)\n\n\ndef _transformer_linears(model: nn.Module) -> "OrderedDict[str, nn.Linear]":\n    result = OrderedDict()\n    for name, module in model.named_modules():\n        if isinstance(module, nn.Linear) and ".layers." in name:\n            result[name] = module\n    if not result:\n        raise ValueError("no transformer Linear modules were found")\n    return result\n\n\ndef _parent_and_child(model: nn.Module, name: str):\n    parts = name.split(".")\n    parent = model\n    for part in parts[:-1]:\n        parent = getattr(parent, part)\n    return parent, parts[-1]\n\n\n@torch.no_grad()\ndef collect_layer_losses(model: nn.Module, input_ids: torch.Tensor,\n                         group_size: int = 128,\n                         max_activation_rows: int = 64) -> Dict[str, Mapping[int, torch.Tensor]]:\n    """Collect per-channel losses during one deterministic calibration pass."""\n    linears = _transformer_linears(model)\n    losses: Dict[str, Mapping[int, torch.Tensor]] = {}\n    handles = []\n\n    def make_hook(name: str):\n        def hook(module, args):\n            if name in losses:\n                return\n            activation = args[0].detach().reshape(-1, module.in_features)\n            activation = activation[:max_activation_rows]\n            layer_losses, _ = estimate_channel_losses(\n                activation, module.weight.detach(), group_size=group_size,\n            )\n            losses[name] = layer_losses\n        return hook\n\n    for name, module in linears.items():\n        if module.in_features % group_size:\n            raise ValueError(f"{name}: in_features must be divisible by group_size")\n        handles.append(module.register_forward_pre_hook(make_hook(name)))\n    try:\n        model(input_ids=input_ids, use_cache=False)\n    finally:\n        for handle in handles:\n            handle.remove()\n    missing = set(linears) - set(losses)\n    if missing:\n        raise RuntimeError(f"calibration did not execute layers: {sorted(missing)[:3]}")\n    return losses\n\n\n@torch.no_grad()\ndef pack_transformer_linears(model: nn.Module, allocations, group_size: int = 128) -> int:\n    """Replace transformer linears with actual packed three-level modules."""\n    linears = _transformer_linears(model)\n    if set(linears) != set(allocations):\n        raise ValueError("allocation names do not match model Linear names")\n    for name, module in linears.items():\n        packed = ThreeLevelLinear.from_weight(\n            module.weight.detach(), allocations[name], group_size=group_size,\n            bias=module.bias,\n        )\n        parent, child = _parent_and_child(model, name)\n        setattr(parent, child, packed)\n    return len(linears)\n\n\ndef _timed_forward(model, input_ids, torch_module, warmup: int, iterations: int):\n    for _ in range(warmup):\n        model(input_ids=input_ids, use_cache=False)\n    torch_module.cuda.synchronize()\n    start = torch_module.cuda.Event(enable_timing=True)\n    end = torch_module.cuda.Event(enable_timing=True)\n    start.record()\n    for _ in range(iterations):\n        model(input_ids=input_ids, use_cache=False)\n    end.record()\n    torch_module.cuda.synchronize()\n    return float(start.elapsed_time(end)) / iterations\n\n\n@torch.no_grad()\ndef run_model_gate(model_id: str, tokenizer, model, calibration_ids: torch.Tensor,\n                   evaluation_ids: torch.Tensor, budget: ThreeLevelBudget | None = None,\n                   target_average_bits: float | None = None,\n                   group_size: int = 128, calibration_rows: int = 64) -> Dict[str, object]:\n    """Quantize a loaded causal LM and return measured quality evidence."""\n    if (budget is None) == (target_average_bits is None):\n        raise ValueError("provide exactly one of budget or target_average_bits")\n    device = next(model.parameters()).device\n    was_training = model.training\n    model.eval()\n    calibration_ids = calibration_ids.to(device)\n    evaluation_ids = evaluation_ids.to(device)\n    if device.type == "cuda":\n        torch.cuda.reset_peak_memory_stats(device)\n    try:\n        reference = model(input_ids=evaluation_ids, labels=evaluation_ids, use_cache=False)\n        reference_loss = float(reference.loss.item())\n        reference_logits = reference.logits[:, -1].float().cpu()\n\n        started = time.perf_counter()\n        losses = collect_layer_losses(model, calibration_ids, group_size, calibration_rows)\n\n        allocation_metadata = {"model_id": model_id, "group_size": group_size}\n        if target_average_bits is None:\n            allocations = allocate_model_channels(\n                losses, budget, alignment=1, metadata=allocation_metadata,\n            )\n            allocation_summary = {"allocator": "fixed_precision_percentages"}\n        else:\n            allocations, allocation_summary = allocate_model_channels_auto(\n                losses, target_average_bits, metadata=allocation_metadata,\n            )\n        packed_layers = pack_transformer_linears(model, allocations, group_size)\n        quantization_seconds = time.perf_counter() - started\n\n        actual = model(input_ids=evaluation_ids, labels=evaluation_ids, use_cache=False)\n        actual_logits = actual.logits[:, -1].float().cpu()\n        quantized_loss = float(actual.loss.item())\n        max_logit_error = float((actual_logits - reference_logits).abs().max().item())\n        mean_logit_error = float((actual_logits - reference_logits).abs().mean().item())\n        deterministic = torch.equal(\n            actual.logits,\n            model(input_ids=evaluation_ids, use_cache=False).logits,\n        )\n        counts = {bit: sum(len(item.indices[bit]) for item in allocations.values())\n                  for bit in (4, 8, 16)}\n        total = sum(counts.values())\n        return {\n            "model_id": model_id,\n            "backend": "packed_reference",\n            "packed_layers": packed_layers,\n            "channel_counts": {str(bit): count for bit, count in counts.items()},\n            "average_weight_bits": sum(bit * counts[bit] for bit in counts) / total,\n            "allocation_summary": allocation_summary,\n            "reference_loss": reference_loss,\n            "reference_perplexity": math.exp(reference_loss),\n            "quantized_loss": quantized_loss,\n            "quantized_perplexity": math.exp(quantized_loss),\n            "loss_delta": quantized_loss - reference_loss,\n            "max_last_token_logit_error": max_logit_error,\n            "mean_last_token_logit_error": mean_logit_error,\n            "deterministic": bool(deterministic),\n            "finite": bool(torch.isfinite(actual.logits).all().item()),\n            "quantization_seconds": quantization_seconds,\n            "allocation_count": len(allocations),\n            "evaluation_tokens": int(evaluation_ids.numel()),\n            "peak_cuda_vram_bytes": (\n                int(torch.cuda.max_memory_allocated(device))\n                if device.type == "cuda" else None\n            ),\n        }\n    finally:\n        if was_training:\n            model.train()\n\n\ndef tokenize_texts(tokenizer, texts: Iterable[str], max_length: int = 128):\n    text = "\\n".join(texts)\n    return tokenizer(text, return_tensors="pt", truncation=True,\n                     max_length=max_length).input_ids\n', 'mixllm/vllm_three_level.py': '"""Version-neutral helpers used by the pinned vLLM three-level patch."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Dict, Iterable, Tuple\n\nfrom mixllm.runtime_capability import RuntimeCapability\n\n\nPINNED_VLLM_VERSION = "0.9.0"\nPINNED_VLLM_COMMIT = "5fbbfe9a4c13094ad72ed3d6b4ef208a7ddc0fd7"\n\n\n@dataclass(frozen=True)\nclass PrecisionPercentages:\n    bit4: int\n    bit8: int\n    bit16: int\n\n    def __post_init__(self) -> None:\n        values = (self.bit4, self.bit8, self.bit16)\n        if any(value < 0 for value in values) or sum(values) != 100:\n            raise ValueError("4/8/16 percentages must sum to 100")\n\n\n@dataclass(frozen=True)\nclass VLLMThreeLevelConfig:\n    group_size: int\n    percentages: PrecisionPercentages\n    backend: str = "auto"\n    allocation_version: int = 1\n\n    @classmethod\n    def from_quantization_config(cls, config: Dict) -> "VLLMThreeLevelConfig":\n        if config.get("quant_method") not in {"mixllm_three_level", "mixllm"}:\n            raise ValueError("checkpoint is not a MixLLM three-level checkpoint")\n        raw = config.get("precision_percentages")\n        if raw is None:\n            raise ValueError("precision_percentages is required for three-level inference")\n        budget = PrecisionPercentages(\n            int(raw.get("4", raw.get(4, 0))),\n            int(raw.get("8", raw.get(8, 0))),\n            int(raw.get("16", raw.get(16, 0))),\n        )\n        group_size = int(config.get("group_size", 128))\n        if group_size != 128:\n            raise ValueError("current MixLLM CUDA ABI requires group_size=128")\n        return cls(\n            group_size=group_size,\n            percentages=budget,\n            backend=str(config.get("backend", "auto")),\n            allocation_version=int(config.get("allocation_version", 1)),\n        )\n\n    def select_backend(self, capability: Tuple[int, int],\n                       sm75_available: bool = False) -> str:\n        return RuntimeCapability(*capability, self.backend, sm75_available).select_backend()\n\n\ndef validate_partition_indices(indices: Dict[int, Iterable[int]], output_size: int) -> None:\n    values = [int(index) for bit in (4, 8, 16) for index in indices.get(bit, ())]\n    if sorted(values) != list(range(output_size)):\n        raise ValueError("vLLM partition indices must cover every output channel exactly once")\n\n\ndef remap_partition_indices_for_tp(\n    indices: Dict[int, Iterable[int]],\n    global_output_size: int,\n    shard_start: int,\n    shard_size: int,\n) -> Dict[int, Tuple[int, ...]]:\n    """Map checkpoint-global output indices to one tensor-parallel shard.\n\n    vLLM loads output-parallel weights as contiguous ranges. The packed tensors\n    retain their precision order, while the operator receives indices local to\n    the current shard so its output remains in original local-channel order.\n    """\n    validate_partition_indices(indices, global_output_size)\n    if shard_start < 0 or shard_size <= 0 or shard_start + shard_size > global_output_size:\n        raise ValueError("invalid tensor-parallel output shard")\n    shard_end = shard_start + shard_size\n    local = {\n        bit: tuple(int(index) - shard_start for index in indices.get(bit, ())\n                   if shard_start <= int(index) < shard_end)\n        for bit in (4, 8, 16)\n    }\n    validate_partition_indices(local, shard_size)\n    return local\n\n\ndef restore_global_partition_indices(\n    local_indices: Dict[int, Iterable[int]], shard_start: int, shard_size: int,\n) -> Dict[int, Tuple[int, ...]]:\n    """Restore checkpoint-global indices after a local shard round trip."""\n    validate_partition_indices(local_indices, shard_size)\n    if shard_start < 0:\n        raise ValueError("shard_start must be non-negative")\n    return {\n        bit: tuple(int(index) + shard_start for index in local_indices.get(bit, ()))\n        for bit in (4, 8, 16)\n    }', 'mixllm/kernels/three_level_sm75.cu': '// Copyright (c) Microsoft Corporation.\n// SPDX-License-Identifier: MIT\n\n#include <ATen/cuda/CUDAContext.h>\n#include <ATen/Functions.h>\n#include <ATen/Tensor.h>\n#include <c10/cuda/CUDAGuard.h>\n#include <c10/cuda/CUDAException.h>\n#include <torch/library.h>\n\n#include <cuda_fp16.h>\n#include <cuda_runtime.h>\n#include <mma.h>\n\nnamespace {\nnamespace wmma = nvcuda::wmma;\n\nconstexpr int kWarpSize = 32;\nconstexpr int kTile = 16;\nconstexpr int kGroupSize = 128;\n// Four 8-lane subwarps share the input stream. This raises decode output\n// parallelism without changing the packed checkpoint ABI.\nconstexpr int kDecodeWarps = 8;\nconstexpr int kDecodeChannelsPerWarp = 4;\nconstexpr int kDecodeSubwarp = kWarpSize / kDecodeChannelsPerWarp;\nconstexpr int kPrefillWarps = 4;\nconstexpr int kPrefillChannels = kPrefillWarps * kTile;\nconstexpr int kReuseRows = 2 * kTile;\n// Four warps per block; each warp quantizes one contiguous 128-element group.\n// This mirrors the upstream MixLLM activation contract without constructing a\n// chain of temporary FP32 tensors through the PyTorch dispatcher.\n__global__ void quantize_activation_sm75_kernel(\n    const __half* input, int8_t* quantized, __half* scales,\n    int rows, int width) {\n#if __CUDA_ARCH__ >= 750\n  const int lane = threadIdx.x % kWarpSize;\n  const int warp = (blockIdx.x * blockDim.x + threadIdx.x) / kWarpSize;\n  const int groups = width / kGroupSize;\n  const int total_groups = rows * groups;\n  if (warp >= total_groups) {\n    return;\n  }\n  const int row = warp / groups;\n  const int group = warp % groups;\n  const int base = row * width + group * kGroupSize;\n  const __half2* input2 = reinterpret_cast<const __half2*>(input + base);\n  const __half2 pair0 = input2[lane * 2];\n  const __half2 pair1 = input2[lane * 2 + 1];\n  const float2 converted0 = __half22float2(pair0);\n  const float2 converted1 = __half22float2(pair1);\n  float values[4] = {converted0.x, converted0.y, converted1.x, converted1.y};\n  float maximum = 0.0f;\n#pragma unroll\n  for (int item = 0; item < 4; ++item) {\n    maximum = fmaxf(maximum, fabsf(values[item]));\n  }\n#pragma unroll\n  for (int offset = 16; offset > 0; offset /= 2) {\n    maximum = fmaxf(maximum, __shfl_down_sync(0xffffffff, maximum, offset));\n  }\n  maximum = __shfl_sync(0xffffffff, maximum, 0);\n  const float scale = fmaxf(maximum / 127.0f, 1.0e-8f);\n  if (lane == 0) {\n    scales[group * rows + row] = __float2half(scale);\n  }\n  uint32_t packed = 0;\n#pragma unroll\n  for (int item = 0; item < 4; ++item) {\n    int value = __float2int_rn(values[item] / scale);\n    value = max(-127, min(127, value));\n    packed |= static_cast<uint32_t>(static_cast<uint8_t>(value)) << (item * 8);\n  }\n  *reinterpret_cast<uint32_t*>(quantized + base + lane * 4) = packed;\n#endif\n}\n\n__global__ void expand_int4_sm75_kernel(\n    const uint8_t* packed, const uint8_t* zeros, int8_t* expanded,\n    int channels, int width) {\n  const int linear = blockIdx.x * blockDim.x + threadIdx.x;\n  const int elements = channels * width;\n  if (linear >= elements) {\n    return;\n  }\n  const int channel = linear / width;\n  const int k = linear % width;\n  const uint8_t byte = packed[channel * (width / 2) + k / 2];\n  const int code = k & 1 ? byte >> 4 : byte & 0x0f;\n  const int groups = width / kGroupSize;\n  expanded[linear] = static_cast<int8_t>(\n      code - static_cast<int>(zeros[channel * groups + k / kGroupSize]));\n}\n\n// SM75 has no Ampere mixed INT8 x INT4 MMA. Prefill reads a cached signed INT8\n// expansion while decode continues to consume the packed checkpoint weights.\n__global__ void three_level_tensorcore_kernel(\n    const __half* input_fp16, const int8_t* input_int8,\n    const __half* scale_act, const int8_t* expanded_int4,\n    const __half* scale_int4,\n    const int32_t* indices_int4, const int8_t* weight_int8,\n    const __half* scale_int8, const int32_t* indices_int8,\n    const __half* weight_fp16, const int32_t* indices_fp16, float* output,\n    int rows, int width, int output_width, int n4, int n8, int n16) {\n#if __CUDA_ARCH__ >= 750\n  const int tiles4 = (n4 + kPrefillChannels - 1) / kPrefillChannels;\n  const int tiles8 = (n8 + kPrefillChannels - 1) / kPrefillChannels;\n  const int tile_id = blockIdx.x;\n  const int row_base = blockIdx.y * kTile;\n  const int warp = threadIdx.x / kWarpSize;\n  const int lane = threadIdx.x % kWarpSize;\n  int precision;\n  int channel_base;\n  if (tile_id < tiles4) {\n    precision = 4;\n    channel_base = tile_id * kPrefillChannels + warp * kTile;\n  } else if (tile_id < tiles4 + tiles8) {\n    precision = 8;\n    channel_base = (tile_id - tiles4) * kPrefillChannels + warp * kTile;\n  } else {\n    precision = 16;\n    channel_base = (tile_id - tiles4 - tiles8) * kPrefillChannels +\n                   warp * kTile;\n  }\n\n  __shared__ __align__(16) int8_t a_int8[kTile * kTile];\n  __shared__ __align__(16) int8_t b_int8[kPrefillWarps][kTile * kTile];\n  __shared__ __align__(16) int accumulator_int[kPrefillWarps][kTile * kTile];\n  __shared__ __align__(16) __half a_fp16[kTile * kTile];\n  __shared__ __align__(16) __half b_fp16[kPrefillWarps][kTile * kTile];\n  __shared__ __align__(16) float accumulator_fp32[kPrefillWarps][kTile * kTile];\n\n  if (precision == 16) {\n    wmma::fragment<wmma::accumulator, kTile, kTile, kTile, float> accumulator;\n    wmma::fill_fragment(accumulator, 0.0f);\n    const bool full_rows = row_base + kTile <= rows;\n    const bool full_channels = channel_base + kTile <= n16;\n    for (int k_base = 0; k_base < width; k_base += kTile) {\n      wmma::fragment<wmma::matrix_a, kTile, kTile, kTile, __half,\n                     wmma::row_major> a;\n      wmma::fragment<wmma::matrix_b, kTile, kTile, kTile, __half,\n                     wmma::col_major> b;\n      for (int linear = threadIdx.x; linear < kTile * kTile;\n           linear += blockDim.x) {\n        const int tile_row = linear / kTile;\n        const int tile_col = linear % kTile;\n        const int row = row_base + tile_row;\n        a_fp16[linear] = row < rows\n            ? input_fp16[row * width + k_base + tile_col]\n            : __float2half(0.0f);\n      }\n      __syncthreads();\n      wmma::load_matrix_sync(a, a_fp16, kTile);\n      if (full_channels) {\n        wmma::load_matrix_sync(\n            b, weight_fp16 + channel_base * width + k_base, width);\n      } else {\n        for (int linear = lane; linear < kTile * kTile;\n             linear += kWarpSize) {\n          const int tile_row = linear / kTile;\n          const int tile_col = linear % kTile;\n          const int channel = channel_base + tile_col;\n          b_fp16[warp][tile_col * kTile + tile_row] = channel < n16\n              ? weight_fp16[channel * width + k_base + tile_row]\n              : __float2half(0.0f);\n        }\n        __syncwarp();\n        wmma::load_matrix_sync(b, b_fp16[warp], kTile);\n      }\n      wmma::mma_sync(accumulator, a, b, accumulator);\n      __syncthreads();\n    }\n    wmma::store_matrix_sync(accumulator_fp32[warp], accumulator, kTile,\n                            wmma::mem_row_major);\n    __syncwarp();\n    for (int linear = lane; linear < kTile * kTile; linear += kWarpSize) {\n      const int row = row_base + linear / kTile;\n      const int local_channel = channel_base + linear % kTile;\n      if (row < rows && local_channel < n16) {\n        output[row * output_width + indices_fp16[local_channel]] =\n            accumulator_fp32[warp][linear];\n      }\n    }\n    return;\n  }\n\n  float scaled_accumulators[kTile * kTile / kWarpSize] = {};\n  const int groups = width / kGroupSize;\n  const int partition_size = precision == 4 ? n4 : n8;\n  const bool full_rows = row_base + kTile <= rows;\n  const bool full_channels = channel_base + kTile <= partition_size;\n  for (int group = 0; group < groups; ++group) {\n    wmma::fragment<wmma::accumulator, kTile, kTile, kTile, int> accumulator;\n    wmma::fill_fragment(accumulator, 0);\n    for (int group_k = 0; group_k < kGroupSize; group_k += kTile) {\n      const int k_base = group * kGroupSize + group_k;\n      wmma::fragment<wmma::matrix_a, kTile, kTile, kTile, signed char,\n                     wmma::row_major> a;\n      wmma::fragment<wmma::matrix_b, kTile, kTile, kTile, signed char,\n                     wmma::col_major> b;\n      for (int linear = threadIdx.x; linear < kTile * kTile;\n           linear += blockDim.x) {\n        const int tile_row = linear / kTile;\n        const int tile_col = linear % kTile;\n        const int row = row_base + tile_row;\n        a_int8[linear] = row < rows\n            ? input_int8[row * width + k_base + tile_col]\n            : int8_t{0};\n      }\n      __syncthreads();\n      wmma::load_matrix_sync(\n          a, reinterpret_cast<signed char*>(a_int8), kTile);\n      if (precision == 8 && full_channels) {\n        wmma::load_matrix_sync(\n            b, reinterpret_cast<const signed char*>(\n                   weight_int8 + channel_base * width + k_base), width);\n      } else if (precision == 4 && full_channels) {\n        wmma::load_matrix_sync(\n            b, reinterpret_cast<const signed char*>(\n                   expanded_int4 + channel_base * width + k_base), width);\n      } else {\n        for (int linear = lane; linear < kTile * kTile;\n             linear += kWarpSize) {\n          const int tile_row = linear / kTile;\n          const int tile_col = linear % kTile;\n          const int channel = channel_base + tile_col;\n          int8_t weight = 0;\n          if (channel < partition_size) {\n            const int k = k_base + tile_row;\n            weight = precision == 4\n                ? expanded_int4[channel * width + k]\n                : weight_int8[channel * width + k];\n          }\n          b_int8[warp][tile_col * kTile + tile_row] = weight;\n        }\n        __syncwarp();\n        wmma::load_matrix_sync(\n            b, reinterpret_cast<signed char*>(b_int8[warp]), kTile);\n      }\n      wmma::mma_sync(accumulator, a, b, accumulator);\n      __syncthreads();\n    }\n    wmma::store_matrix_sync(accumulator_int[warp], accumulator, kTile,\n                            wmma::mem_row_major);\n    __syncwarp();\n    for (int linear = lane, item = 0; linear < kTile * kTile;\n         linear += kWarpSize, ++item) {\n      const int tile_row = linear / kTile;\n      const int local_channel = channel_base + linear % kTile;\n      if (row_base + tile_row < rows && local_channel < partition_size) {\n        const float activation_scale =\n            __half2float(scale_act[group * rows + row_base + tile_row]);\n        const __half weight_scale = precision == 4\n            ? scale_int4[local_channel * groups + group]\n            : scale_int8[local_channel * groups + group];\n        scaled_accumulators[item] += static_cast<float>(accumulator_int[warp][linear]) *\n                                     activation_scale * __half2float(weight_scale);\n      }\n    }\n    __syncwarp();\n  }\n  for (int linear = lane, item = 0; linear < kTile * kTile;\n       linear += kWarpSize, ++item) {\n    const int row = row_base + linear / kTile;\n    const int local_channel = channel_base + linear % kTile;\n    if (row < rows && local_channel < partition_size) {\n      const int output_channel = precision == 4\n          ? indices_int4[local_channel] : indices_int8[local_channel];\n      output[row * output_width + output_channel] = scaled_accumulators[item];\n    }\n  }\n#endif\n}\n\n// Balanced prefill variant. Eight warps form a 2x4 grid of 16x16 WMMA tiles.\n// Each 32x16 A panel and 16x64 B panel is loaded once per CTA/K step.\n__global__ void three_level_tensorcore_reuse_kernel(\n    const __half* input_fp16, const int8_t* input_int8,\n    const __half* scale_act, const int8_t* expanded_int4,\n    const __half* scale_int4,\n    const int32_t* indices_int4, const int8_t* weight_int8,\n    const __half* scale_int8, const int32_t* indices_int8,\n    const __half* weight_fp16, const int32_t* indices_fp16, float* output,\n    int rows, int width, int output_width, int n4, int n8, int n16) {\n#if __CUDA_ARCH__ >= 750\n  const int warp = threadIdx.x / kWarpSize;\n  const int lane = threadIdx.x % kWarpSize;\n  const int warp_row = warp / 4;\n  const int warp_channel = warp % 4;\n  const int row_tile_base = blockIdx.y * kReuseRows;\n  const int row_base = row_tile_base + warp_row * kTile;\n  const int precision_tile = blockIdx.x;\n  const int tiles4 = (n4 + kPrefillChannels - 1) / kPrefillChannels;\n  const int tiles8 = (n8 + kPrefillChannels - 1) / kPrefillChannels;\n  int precision = 16;\n  int cta_channel_base = 0;\n  int partition_size = n16;\n  if (precision_tile < tiles4) {\n    precision = 4;\n    cta_channel_base = precision_tile * kPrefillChannels;\n    partition_size = n4;\n  } else if (precision_tile < tiles4 + tiles8) {\n    precision = 8;\n    cta_channel_base = (precision_tile - tiles4) * kPrefillChannels;\n    partition_size = n8;\n  } else {\n    cta_channel_base = (precision_tile - tiles4 - tiles8) * kPrefillChannels;\n  }\n  const int channel_base = cta_channel_base + warp_channel * kTile;\n  __shared__ __align__(16) int8_t a_int8[2][kTile * kTile];\n  __shared__ __align__(16) int8_t b_int8[4][kTile * kTile];\n  __shared__ __align__(16) int accumulator_int[8][kTile * kTile];\n  __shared__ __align__(16) __half a_fp16[2][kTile * kTile];\n  __shared__ __align__(16) __half b_fp16[4][kTile * kTile];\n  __shared__ __align__(16) float accumulator_fp32[8][kTile * kTile];\n  const int groups = width / kGroupSize;\n  if (precision == 16) {\n    wmma::fragment<wmma::accumulator, kTile, kTile, kTile, float> acc;\n    wmma::fill_fragment(acc, 0.0f);\n    for (int k_base = 0; k_base < width; k_base += kTile) {\n      for (int linear = threadIdx.x; linear < 2 * kTile * kTile;\n           linear += blockDim.x) {\n        const int tile = linear / (kTile * kTile);\n        const int item = linear % (kTile * kTile);\n        const int r = row_tile_base + tile * kTile + item / kTile;\n        a_fp16[tile][item] = r < rows\n            ? input_fp16[r * width + k_base + item % kTile]\n            : __float2half(0.0f);\n      }\n      for (int linear = threadIdx.x; linear < 4 * kTile * kTile;\n           linear += blockDim.x) {\n        const int tile = linear / (kTile * kTile);\n        const int item = linear % (kTile * kTile);\n        const int c = cta_channel_base + tile * kTile + item / kTile;\n        b_fp16[tile][item] = c < partition_size\n            ? weight_fp16[c * width + k_base + item % kTile]\n            : __float2half(0.0f);\n      }\n      __syncthreads();\n      wmma::fragment<wmma::matrix_a, kTile, kTile, kTile, __half,\n                     wmma::row_major> a;\n      wmma::fragment<wmma::matrix_b, kTile, kTile, kTile, __half,\n                     wmma::col_major> b;\n      wmma::load_matrix_sync(a, a_fp16[warp_row], kTile);\n      wmma::load_matrix_sync(b, b_fp16[warp_channel], kTile);\n      wmma::mma_sync(acc, a, b, acc);\n      __syncthreads();\n    }\n    wmma::store_matrix_sync(accumulator_fp32[warp], acc, kTile,\n                            wmma::mem_row_major);\n    __syncwarp();\n    for (int linear = lane; linear < kTile * kTile; linear += kWarpSize) {\n      const int r = row_base + linear / kTile;\n      const int c = channel_base + linear % kTile;\n      if (r < rows && c < partition_size)\n        output[r * output_width + indices_fp16[c]] =\n            accumulator_fp32[warp][linear];\n    }\n    return;\n  }\n  float scaled[kTile * kTile / kWarpSize] = {};\n  for (int group = 0; group < groups; ++group) {\n    for (int group_k = 0; group_k < kGroupSize; group_k += kTile) {\n      const int k_base = group * kGroupSize + group_k;\n      for (int linear = threadIdx.x; linear < 2 * kTile * kTile;\n           linear += blockDim.x) {\n        const int tile = linear / (kTile * kTile);\n        const int item = linear % (kTile * kTile);\n        const int r = row_tile_base + tile * kTile + item / kTile;\n        a_int8[tile][item] = r < rows\n            ? input_int8[r * width + k_base + item % kTile] : 0;\n      }\n      for (int linear = threadIdx.x; linear < 4 * kTile * kTile;\n           linear += blockDim.x) {\n        const int tile = linear / (kTile * kTile);\n        const int item = linear % (kTile * kTile);\n        const int c = cta_channel_base + tile * kTile + item / kTile;\n        int8_t value = 0;\n        if (c < partition_size) {\n          const int k = k_base + item % kTile;\n          value = precision == 8 ? weight_int8[c * width + k]\n                                 : expanded_int4[c * width + k];\n        }\n        b_int8[tile][item] = value;\n      }\n      __syncthreads();\n      wmma::fragment<wmma::accumulator, kTile, kTile, kTile, int> acc;\n      wmma::fill_fragment(acc, 0);\n      wmma::fragment<wmma::matrix_a, kTile, kTile, kTile, signed char,\n                     wmma::row_major> a;\n      wmma::fragment<wmma::matrix_b, kTile, kTile, kTile, signed char,\n                     wmma::col_major> b;\n      wmma::load_matrix_sync(a, a_int8[warp_row], kTile);\n      wmma::load_matrix_sync(b, b_int8[warp_channel], kTile);\n      wmma::mma_sync(acc, a, b, acc);\n      __syncthreads();\n      wmma::store_matrix_sync(accumulator_int[warp], acc, kTile,\n                              wmma::mem_row_major);\n      __syncwarp();\n      for (int linear = lane; linear < kTile * kTile; linear += kWarpSize) {\n        const int r = row_base + linear / kTile;\n        const int c = channel_base + linear % kTile;\n        if (r < rows && c < partition_size) {\n          const float as = __half2float(scale_act[group * rows + r]);\n          const float ws = __half2float(precision == 4\n              ? scale_int4[c * groups + group] : scale_int8[c * groups + group]);\n          scaled[linear / kWarpSize] +=\n              static_cast<float>(accumulator_int[warp][linear]) * as * ws;\n        }\n      }\n      __syncthreads();\n    }\n  }\n  for (int linear = lane; linear < kTile * kTile; linear += kWarpSize) {\n    const int r = row_base + linear / kTile;\n    const int c = channel_base + linear % kTile;\n    if (r < rows && c < partition_size)\n      output[r * output_width + (precision == 4 ? indices_int4[c] : indices_int8[c])] = scaled[linear / kWarpSize];\n  }\n#endif\n}\n\n// Decode is bandwidth-bound and WMMA would compute 15 unused rows for M=1.\n// Four 8-lane subwarps share one warp; each subwarp owns one output channel and\n// performs four dp4a operations per 128-element activation group. This avoids\n// assigning a full warp and five reduction steps to every scalar output.\n__global__ void three_level_decode_kernel(\n    const __half* input_fp16, const int8_t* input_int8,\n    const __half* scale_act, const uint8_t* packed_int4,\n    const __half* scale_int4, const uint8_t* zero_int4,\n    const int32_t* indices_int4, const int8_t* weight_int8,\n    const __half* scale_int8, const int32_t* indices_int8,\n    const __half* weight_fp16, const int32_t* indices_fp16, float* output,\n    int width, int output_width, int n4, int n8, int n16) {\n#if __CUDA_ARCH__ >= 750\n  const int lane = threadIdx.x & (kWarpSize - 1);\n  const int local_warp = threadIdx.x / kWarpSize;\n  const int subwarp = lane / kDecodeSubwarp;\n  const int sublane = lane & (kDecodeSubwarp - 1);\n  const int channel = blockIdx.x * (kDecodeWarps * kDecodeChannelsPerWarp) +\n                      local_warp * kDecodeChannelsPerWarp + subwarp;\n  if (channel >= output_width) {\n    return;\n  }\n\n  const int groups = width / kGroupSize;\n  const unsigned int subwarp_mask = __activemask();\n  float result = 0.0f;\n  int output_channel;\n  if (channel < n4) {\n    output_channel = indices_int4[channel];\n    const uint8_t* weights = packed_int4 + channel * (width / 2);\n    for (int group = 0; group < groups; ++group) {\n      const int base = group * kGroupSize;\n      const int zero = static_cast<int>(zero_int4[channel * groups + group]);\n      int accumulator = 0;\n#pragma unroll\n      for (int offset = sublane * 4; offset < kGroupSize;\n           offset += kDecodeSubwarp * 4) {\n        const int k = base + offset;\n        const uint8_t packed01 = __ldg(weights + k / 2);\n        const uint8_t packed23 = __ldg(weights + k / 2 + 1);\n        const int w0 = static_cast<int>(packed01 & 0x0f) - zero;\n        const int w1 = static_cast<int>(packed01 >> 4) - zero;\n        const int w2 = static_cast<int>(packed23 & 0x0f) - zero;\n        const int w3 = static_cast<int>(packed23 >> 4) - zero;\n        const int packed_weights =\n            (w0 & 0xff) | ((w1 & 0xff) << 8) |\n            ((w2 & 0xff) << 16) | ((w3 & 0xff) << 24);\n        const int activations = __ldg(reinterpret_cast<const int*>(input_int8 + k));\n        accumulator = __dp4a(packed_weights, activations, accumulator);\n      }\n      for (int delta = kDecodeSubwarp / 2; delta > 0; delta >>= 1) {\n        accumulator += __shfl_down_sync(subwarp_mask, accumulator, delta,\n                                         kDecodeSubwarp);\n      }\n      if (sublane == 0) {\n        result += static_cast<float>(accumulator) *\n                  __half2float(__ldg(scale_act + group)) *\n                  __half2float(__ldg(scale_int4 + channel * groups + group));\n      }\n    }\n  } else if (channel < n4 + n8) {\n    const int local_channel = channel - n4;\n    output_channel = indices_int8[local_channel];\n    const int8_t* weights = weight_int8 + local_channel * width;\n    for (int group = 0; group < groups; ++group) {\n      const int base = group * kGroupSize;\n      int accumulator = 0;\n#pragma unroll\n      for (int offset = sublane * 4; offset < kGroupSize;\n           offset += kDecodeSubwarp * 4) {\n        const int k = base + offset;\n        const int packed_weights = __ldg(reinterpret_cast<const int*>(weights + k));\n        const int activations = __ldg(reinterpret_cast<const int*>(input_int8 + k));\n        accumulator = __dp4a(packed_weights, activations, accumulator);\n      }\n      for (int delta = kDecodeSubwarp / 2; delta > 0; delta >>= 1) {\n        accumulator += __shfl_down_sync(subwarp_mask, accumulator, delta,\n                                         kDecodeSubwarp);\n      }\n      if (sublane == 0) {\n        result += static_cast<float>(accumulator) *\n                  __half2float(__ldg(scale_act + group)) *\n                  __half2float(scale_int8[local_channel * groups + group]);\n      }\n    }\n  } else {\n    const int local_channel = channel - n4 - n8;\n    output_channel = indices_fp16[local_channel];\n    const __half2* input2 = reinterpret_cast<const __half2*>(input_fp16);\n    const __half2* weights2 = reinterpret_cast<const __half2*>(\n        weight_fp16 + local_channel * width);\n    float accumulator = 0.0f;\n    for (int k2 = sublane; k2 < width / 2; k2 += kDecodeSubwarp) {\n      const float2 product = __half22float2(__hmul2(input2[k2], weights2[k2]));\n      accumulator += product.x + product.y;\n    }\n    for (int delta = kDecodeSubwarp / 2; delta > 0; delta >>= 1) {\n      accumulator += __shfl_down_sync(subwarp_mask, accumulator, delta,\n                                       kDecodeSubwarp);\n    }\n    if (sublane == 0) {\n      result = accumulator;\n    }\n  }\n  if (sublane == 0) {\n    output[output_channel] = result;\n  }\n#endif\n}\n\nvoid check_cuda_contiguous(const at::Tensor& tensor, const char* name) {\n  TORCH_CHECK(tensor.is_cuda(), name, " must be a CUDA tensor");\n  TORCH_CHECK(tensor.is_contiguous(), name, " must be contiguous");\n}\n\nvoid check_same_device(const at::Tensor& input, const at::Tensor& tensor,\n                       const char* name) {\n  TORCH_CHECK(tensor.device() == input.device(), name,\n              " must be on the same CUDA device as input_fp16");\n}\n\nstd::tuple<at::Tensor, at::Tensor> quantize_activation_sm75(\n    const at::Tensor& input) {\n  check_cuda_contiguous(input, "input");\n  TORCH_CHECK(input.dim() == 2 && input.scalar_type() == at::kHalf,\n              "input must be a contiguous float16 matrix");\n  const int rows = input.size(0);\n  const int width = input.size(1);\n  TORCH_CHECK(width % kGroupSize == 0,\n              "SM75 activation quantization requires K divisible by 128");\n  c10::cuda::CUDAGuard device_guard(input.device());\n  auto quantized = at::empty(input.sizes(), input.options().dtype(at::kChar));\n  auto scales = at::empty({width / kGroupSize, rows},\n                          input.options().dtype(at::kHalf));\n  constexpr int threads = 128;\n  constexpr int warps_per_block = threads / kWarpSize;\n  const int total_groups = rows * (width / kGroupSize);\n  if (total_groups == 0) {\n    return std::make_tuple(quantized, scales);\n  }\n  const int blocks = (total_groups + warps_per_block - 1) / warps_per_block;\n  quantize_activation_sm75_kernel<<<blocks, threads, 0,\n      at::cuda::getCurrentCUDAStream()>>>(\n      reinterpret_cast<const __half*>(input.data_ptr<at::Half>()),\n      quantized.data_ptr<int8_t>(),\n      reinterpret_cast<__half*>(scales.data_ptr<at::Half>()), rows, width);\n  C10_CUDA_KERNEL_LAUNCH_CHECK();\n  return std::make_tuple(quantized, scales);\n}\n\nat::Tensor three_level_linear_v2_core(\n    const at::Tensor& input_fp16, const at::Tensor& input_int8,\n    const at::Tensor& scale_act, const at::Tensor& weight_int4,\n    const at::Tensor& expanded_int4, const at::Tensor& scale_int4,\n    const at::Tensor& zero_int4,\n    const at::Tensor& indices_int4, const at::Tensor& weight_int8,\n    const at::Tensor& scale_int8, const at::Tensor& indices_int8,\n    const at::Tensor& weight_fp16, const at::Tensor& indices_fp16) {\n  const at::Tensor* tensors[] = {&input_int8, &scale_act, &weight_int4,\n      &expanded_int4, &scale_int4, &zero_int4, &indices_int4, &weight_int8,\n      &scale_int8, &indices_int8, &weight_fp16, &indices_fp16};\n  const char* names[] = {"input_int8", "scale_act", "weight_int4",\n      "expanded_int4", "scale_int4", "zero_int4", "indices_int4",\n      "weight_int8", "scale_int8", "indices_int8", "weight_fp16",\n      "indices_fp16"};\n  check_cuda_contiguous(input_fp16, "input_fp16");\n  check_cuda_contiguous(input_int8, "input_int8");\n  check_cuda_contiguous(scale_act, "scale_act");\n  check_cuda_contiguous(weight_int4, "weight_int4");\n  check_cuda_contiguous(expanded_int4, "expanded_int4");\n  check_cuda_contiguous(scale_int4, "scale_int4");\n  check_cuda_contiguous(zero_int4, "zero_int4");\n  check_cuda_contiguous(indices_int4, "indices_int4");\n  check_cuda_contiguous(weight_int8, "weight_int8");\n  check_cuda_contiguous(scale_int8, "scale_int8");\n  check_cuda_contiguous(indices_int8, "indices_int8");\n  check_cuda_contiguous(weight_fp16, "weight_fp16");\n  check_cuda_contiguous(indices_fp16, "indices_fp16");\n  for (int i = 0; i < 12; ++i) {\n    check_same_device(input_fp16, *tensors[i], names[i]);\n  }\n  TORCH_CHECK(input_fp16.dim() == 2 && input_fp16.scalar_type() == at::kHalf,\n              "input_fp16 must be a float16 matrix");\n  TORCH_CHECK(input_int8.sizes() == input_fp16.sizes() &&\n                  input_int8.scalar_type() == at::kChar,\n              "input_int8 must match input_fp16 and have dtype int8");\n  TORCH_CHECK(scale_act.scalar_type() == at::kHalf && scale_act.dim() == 2,\n              "scale_act must be a float16 matrix");\n  TORCH_CHECK(weight_int4.scalar_type() == at::kByte &&\n                  zero_int4.scalar_type() == at::kByte,\n              "INT4 codes and zero points must be uint8");\n  TORCH_CHECK(weight_int8.scalar_type() == at::kChar,\n              "weight_int8 must have dtype int8");\n  TORCH_CHECK(expanded_int4.scalar_type() == at::kChar,\n              "expanded_int4 must have dtype int8");\n  TORCH_CHECK(scale_int4.scalar_type() == at::kHalf &&\n                  scale_int8.scalar_type() == at::kHalf &&\n                  weight_fp16.scalar_type() == at::kHalf,\n              "weights/scales must use the checkpoint ABI dtypes");\n  TORCH_CHECK(indices_int4.scalar_type() == at::kInt &&\n                  indices_int8.scalar_type() == at::kInt &&\n                  indices_fp16.scalar_type() == at::kInt,\n              "all channel indices must be int32");\n\n  const int rows = input_fp16.size(0);\n  const int width = input_fp16.size(1);\n  const int n4 = indices_int4.numel();\n  const int n8 = indices_int8.numel();\n  const int n16 = indices_fp16.numel();\n  const int output_width = n4 + n8 + n16;\n  TORCH_CHECK(output_width > 0, "at least one precision partition is required");\n  TORCH_CHECK(width % kGroupSize == 0,\n              "SM75 Tensor Core backend requires K divisible by 128");\n  TORCH_CHECK(scale_act.size(0) == width / kGroupSize &&\n                  scale_act.size(1) == rows,\n              "scale_act must have shape [K/128, rows]");\n  TORCH_CHECK(weight_int4.size(0) == n4 && weight_int4.size(1) == width / 2,\n              "invalid packed INT4 weight shape");\n  TORCH_CHECK(rows == 1 ||\n                  (expanded_int4.dim() == 2 && expanded_int4.size(0) == n4 &&\n                   expanded_int4.size(1) == width),\n              "expanded_int4 must have shape [n4, K] for prefill");\n  TORCH_CHECK(weight_int8.size(0) == n8 && weight_int8.size(1) == width,\n              "invalid INT8 weight shape");\n  TORCH_CHECK(weight_fp16.size(0) == n16 && weight_fp16.size(1) == width,\n              "invalid FP16 weight shape");\n  const int groups = width / kGroupSize;\n  TORCH_CHECK(scale_int4.size(0) == n4 && scale_int4.size(1) == groups &&\n                  zero_int4.size(0) == n4 && zero_int4.size(1) == groups,\n              "invalid INT4 metadata shape");\n  TORCH_CHECK(scale_int8.size(0) == n8 && scale_int8.size(1) == groups,\n              "invalid INT8 scale shape");\n\n  c10::cuda::CUDAGuard device_guard(input_fp16.device());\n  auto output = at::empty({rows, output_width}, input_fp16.options().dtype(at::kFloat));\n  if (rows == 0) {\n    return output;\n  }\n  auto stream = at::cuda::getCurrentCUDAStream();\n  if (rows == 1) {\n    const int channels_per_block = kDecodeWarps * kDecodeChannelsPerWarp;\n    const int blocks = (output_width + channels_per_block - 1) /\n                       channels_per_block;\n    three_level_decode_kernel<<<blocks, kDecodeWarps * kWarpSize, 0, stream>>>(\n        reinterpret_cast<const __half*>(input_fp16.data_ptr<at::Half>()),\n        input_int8.data_ptr<int8_t>(),\n        reinterpret_cast<const __half*>(scale_act.data_ptr<at::Half>()),\n        weight_int4.data_ptr<uint8_t>(),\n        reinterpret_cast<const __half*>(scale_int4.data_ptr<at::Half>()),\n        zero_int4.data_ptr<uint8_t>(), indices_int4.data_ptr<int32_t>(),\n        weight_int8.data_ptr<int8_t>(),\n        reinterpret_cast<const __half*>(scale_int8.data_ptr<at::Half>()),\n        indices_int8.data_ptr<int32_t>(),\n        reinterpret_cast<const __half*>(weight_fp16.data_ptr<at::Half>()),\n        indices_fp16.data_ptr<int32_t>(), output.data_ptr<float>(), width,\n        output_width, n4, n8, n16);\n  } else {\n    const int channel_tiles =\n        (n4 + kPrefillChannels - 1) / kPrefillChannels +\n        (n8 + kPrefillChannels - 1) / kPrefillChannels +\n        (n16 + kPrefillChannels - 1) / kPrefillChannels;\n    const dim3 grid(channel_tiles, (rows + 15) / 16);\n    three_level_tensorcore_kernel<<<grid, kPrefillWarps * kWarpSize, 0, stream>>>(\n      reinterpret_cast<const __half*>(input_fp16.data_ptr<at::Half>()),\n      input_int8.data_ptr<int8_t>(),\n      reinterpret_cast<const __half*>(scale_act.data_ptr<at::Half>()),\n      expanded_int4.data_ptr<int8_t>(),\n      reinterpret_cast<const __half*>(scale_int4.data_ptr<at::Half>()),\n      indices_int4.data_ptr<int32_t>(),\n      weight_int8.data_ptr<int8_t>(),\n      reinterpret_cast<const __half*>(scale_int8.data_ptr<at::Half>()),\n      indices_int8.data_ptr<int32_t>(),\n      reinterpret_cast<const __half*>(weight_fp16.data_ptr<at::Half>()),\n       indices_fp16.data_ptr<int32_t>(), output.data_ptr<float>(), rows, width,\n       output_width, n4, n8, n16);\n  }\n  C10_CUDA_KERNEL_LAUNCH_CHECK();\n  return output;\n}\nvoid validate_partition(const at::Tensor& indices_int4,\n                        const at::Tensor& indices_int8,\n                        const at::Tensor& indices_fp16) {\n  const int64_t output_width = indices_int4.numel() + indices_int8.numel() +\n                               indices_fp16.numel();\n  auto complete = at::cat({indices_int4, indices_int8, indices_fp16});\n  auto expected = at::arange(output_width, indices_int4.options());\n  TORCH_CHECK(std::get<0>(complete.sort()).equal(expected),\n              "4/8/16 indices must form a complete output partition");\n}\n\nat::Tensor three_level_linear_v2_cuda(\n    const at::Tensor& input_fp16, const at::Tensor& input_int8,\n    const at::Tensor& scale_act, const at::Tensor& weight_int4,\n    const at::Tensor& expanded_int4, const at::Tensor& scale_int4,\n    const at::Tensor& zero_int4,\n    const at::Tensor& indices_int4, const at::Tensor& weight_int8,\n    const at::Tensor& scale_int8, const at::Tensor& indices_int8,\n    const at::Tensor& weight_fp16, const at::Tensor& indices_fp16) {\n  validate_partition(indices_int4, indices_int8, indices_fp16);\n  return three_level_linear_v2_core(\n      input_fp16, input_int8, scale_act, weight_int4, expanded_int4, scale_int4,\n      zero_int4, indices_int4, weight_int8, scale_int8, indices_int8,\n      weight_fp16, indices_fp16);\n}\n\nat::Tensor three_level_linear_legacy_cuda(\n    const at::Tensor& input, const at::Tensor& weight_int4,\n    const at::Tensor& scale_int4, const at::Tensor& zero_int4,\n    const at::Tensor& indices_int4, const at::Tensor& weight_int8,\n    const at::Tensor& scale_int8, const at::Tensor& indices_int8,\n    const at::Tensor& weight_fp16, const at::Tensor& indices_fp16,\n    int64_t group_size) {\n  TORCH_CHECK(group_size == kGroupSize,\n              "legacy SM75 adapter requires group_size=128");\n  check_cuda_contiguous(input, "input");\n  TORCH_CHECK(input.dim() == 2 &&\n                  (input.scalar_type() == at::kHalf ||\n                   input.scalar_type() == at::kFloat),\n              "input must be a float16 or float32 matrix");\n  auto input_fp16 = input.scalar_type() == at::kHalf\n      ? input : input.to(at::kHalf);\n  auto activation = quantize_activation_sm75(input_fp16);\n  auto expanded_int4 = input_fp16.size(0) == 1\n      ? weight_int8\n      : at::empty({weight_int4.size(0), input_fp16.size(1)},\n                  weight_int8.options());\n  const int elements = expanded_int4.numel();\n  if (input_fp16.size(0) > 1 && elements > 0) {\n    constexpr int threads = 256;\n    expand_int4_sm75_kernel<<<(elements + threads - 1) / threads, threads, 0,\n        at::cuda::getCurrentCUDAStream()>>>(\n        weight_int4.data_ptr<uint8_t>(), zero_int4.data_ptr<uint8_t>(),\n        expanded_int4.data_ptr<int8_t>(), weight_int4.size(0), input_fp16.size(1));\n    C10_CUDA_KERNEL_LAUNCH_CHECK();\n  }\n  validate_partition(indices_int4, indices_int8, indices_fp16);\n  return three_level_linear_v2_core(\n      input_fp16, std::get<0>(activation), std::get<1>(activation), weight_int4,\n      expanded_int4, scale_int4, zero_int4, indices_int4, weight_int8,\n      scale_int8, indices_int8, weight_fp16, indices_fp16);\n}\n\n}  // namespace\n\nTORCH_LIBRARY(mixllm_sm75, m) {\n  m.def("quantize_activation(Tensor input) -> (Tensor, Tensor)");\n  m.def("_three_level_linear_v2_unchecked(Tensor input_fp16, Tensor input_int8, "\n        "Tensor scale_act, Tensor weight_int4, Tensor expanded_int4, "\n        "Tensor scale_int4, "\n        "Tensor zero_int4, Tensor indices_int4, Tensor weight_int8, "\n        "Tensor scale_int8, Tensor indices_int8, Tensor weight_fp16, "\n        "Tensor indices_fp16) -> Tensor");\n  m.def("three_level_linear_v2(Tensor input_fp16, Tensor input_int8, "\n        "Tensor scale_act, Tensor weight_int4, Tensor expanded_int4, "\n        "Tensor scale_int4, "\n        "Tensor zero_int4, Tensor indices_int4, Tensor weight_int8, "\n        "Tensor scale_int8, Tensor indices_int8, Tensor weight_fp16, "\n        "Tensor indices_fp16) -> Tensor");\n  m.def("three_level_linear(Tensor input, Tensor weight_int4, Tensor scale_int4, "\n        "Tensor zero_int4, Tensor indices_int4, Tensor weight_int8, "\n        "Tensor scale_int8, Tensor indices_int8, Tensor weight_fp16, "\n        "Tensor indices_fp16, int group_size) -> Tensor");\n}\n\nTORCH_LIBRARY_IMPL(mixllm_sm75, CUDA, m) {\n  m.impl("quantize_activation", &quantize_activation_sm75);\n  m.impl("_three_level_linear_v2_unchecked", &three_level_linear_v2_core);\n  m.impl("three_level_linear_v2", &three_level_linear_v2_cuda);\n  m.impl("three_level_linear", &three_level_linear_legacy_cuda);\n}\n', 'mixllm/test/test_three_level.py': 'import tempfile\nimport unittest\nfrom pathlib import Path\n\nimport torch\n\nfrom mixllm.quantization.three_level import (\n    ThreeLevelAllocation,\n    ThreeLevelBudget,\n    allocate_channels,\n    allocate_model_channels,\n    allocate_model_channels_auto,\n    estimate_channel_losses,\n    fake_linear,\n)\nfrom mixllm.nn.modules.three_level_linear import ThreeLevelLinear\n\n\nclass ThreeLevelAllocationTest(unittest.TestCase):\n\n    def test_uses_marginal_benefit_for_each_upgrade(self):\n        losses = {\n            4: [100.0, 10.0, 9.0, 8.0],\n            8: [99.0, 0.0, 8.0, 7.0],\n            16: [0.0, 0.0, 8.0, 7.0],\n        }\n        result = allocate_channels(losses, ThreeLevelBudget(50, 25, 25))\n\n        self.assertEqual(result.indices[16], (0,))\n        self.assertEqual(result.indices[8], (1,))\n        self.assertEqual(result.indices[4], (2, 3))\n\n    def test_allows_empty_partitions(self):\n        losses = {4: [3.0, 2.0], 8: [1.0, 1.0], 16: [0.0, 0.0]}\n        result = allocate_channels(losses, ThreeLevelBudget(0, 0, 100))\n\n        self.assertEqual(result.indices[4], ())\n        self.assertEqual(result.indices[8], ())\n        self.assertEqual(result.indices[16], (0, 1))\n\n    def test_alignment_preserves_complete_partition(self):\n        losses = {bit: torch.arange(10, dtype=torch.float32) / bit for bit in (4, 8, 16)}\n        result = allocate_channels(losses, ThreeLevelBudget(60, 20, 20), alignment=2)\n\n        result.verify(10)\n        self.assertEqual({bit: len(result.indices[bit]) for bit in (4, 8, 16)},\n                         {4: 6, 8: 2, 16: 2})\n\n    def test_json_round_trip(self):\n        losses = {4: [3.0, 2.0], 8: [1.0, 1.0], 16: [0.0, 0.0]}\n        result = allocate_channels(losses, ThreeLevelBudget(50, 0, 50))\n        with tempfile.TemporaryDirectory() as directory:\n            path = Path(directory) / "allocation.json"\n            result.to_json(path)\n            loaded = ThreeLevelAllocation.from_json(path)\n\n        self.assertEqual(loaded, result)\n\n    def test_rejects_invalid_inputs(self):\n        with self.assertRaises(ValueError):\n            ThreeLevelBudget(90, 20, 0)\n        with self.assertRaises(ValueError):\n            allocate_channels({4: [], 8: [], 16: []}, ThreeLevelBudget(100, 0, 0))\n        with self.assertRaises(ValueError):\n            allocate_channels({4: [1], 8: [1, 2], 16: [0]},\n                              ThreeLevelBudget(100, 0, 0))\n\n    def test_global_allocator_preserves_counts_and_layer_minimums(self):\n        losses = {\n            "a": {4: [10, 9, 8, 7], 8: [1, 1, 1, 1], 16: [0, 0, 0, 0]},\n            "b": {4: [2, 2, 2, 2], 8: [1, 1, 1, 1], 16: [0, 0, 0, 0]},\n        }\n        result = allocate_model_channels(\n            losses, ThreeLevelBudget(50, 25, 25), layer_minimums={"b": 2})\n\n        counts = {bit: sum(len(value.indices[bit]) for value in result.values())\n                  for bit in (4, 8, 16)}\n        self.assertEqual(counts, {4: 4, 8: 2, 16: 2})\n        self.assertGreaterEqual(len(result["b"].indices[8]) +\n                                len(result["b"].indices[16]), 2)\n        for name, allocation in result.items():\n            allocation.verify(4)\n            self.assertEqual(allocation.metadata["layer_name"], name)\n\n    def test_auto_allocator_is_deterministic_across_layer_order(self):\n        losses = {\n            "a": {4: [10, 10], 8: [9, 9], 16: [9, 9]},\n            "b": {4: [10, 10], 8: [9, 9], 16: [9, 9]},\n        }\n\n        first, first_summary = allocate_model_channels_auto(losses, 6)\n        second, second_summary = allocate_model_channels_auto(\n            dict(reversed(list(losses.items()))), 6)\n\n        self.assertEqual(first, second)\n        self.assertEqual(first_summary, second_summary)\n        self.assertEqual(first["a"].indices[8], (0, 1))\n        self.assertEqual(first["b"].indices[4], (0, 1))\n\n    def test_auto_allocator_honors_bit_budget_and_alignment(self):\n        losses = {\n            "a": {4: [12, 11, 10, 9], 8: [6, 6, 6, 6], 16: [0, 0, 0, 0]},\n            "b": {4: [8, 7, 6, 5], 8: [4, 4, 4, 4], 16: [0, 0, 0, 0]},\n        }\n\n        result, summary = allocate_model_channels_auto(\n            losses, 10, alignment=2, metadata={"model_id": "tiny"})\n\n        counts = {bit: sum(len(item.indices[bit]) for item in result.values())\n                  for bit in (4, 8, 16)}\n        self.assertEqual(counts, {int(bit): count\n                                 for bit, count in summary["channel_counts"].items()})\n        for allocation in result.values():\n            self.assertEqual(len(allocation.indices[8]) % 2, 0)\n            self.assertEqual(len(allocation.indices[16]) % 2, 0)\n        self.assertLessEqual(summary["achieved_average_bits"], 10)\n        self.assertEqual(summary["unused_bit_budget"], 0)\n        self.assertEqual(result["a"].metadata["model_id"], "tiny")\n        for allocation in result.values():\n            allocation.verify(4)\n\n    def test_auto_allocator_uses_per_layer_aligned_upgrade_blocks(self):\n        losses = {\n            "a": {4: [100, 0], 8: [0, 0], 16: [0, 0]},\n            "b": {4: [99, 98], 8: [0, 0], 16: [0, 0]},\n        }\n\n        result, _ = allocate_model_channels_auto(losses, 6, alignment=2)\n\n        self.assertEqual(result["a"].indices[4], (0, 1))\n        self.assertEqual(result["b"].indices[8], (0, 1))\n\n    def test_auto_allocator_allows_small_unaligned_totals(self):\n        losses = {\n            "small": {4: [100], 8: [0], 16: [0]},\n            "full": {4: [2, 1], 8: [0, 0], 16: [0, 0]},\n        }\n\n        result, summary = allocate_model_channels_auto(losses, 8, alignment=2)\n\n        self.assertEqual(result["small"].indices[4], (0,))\n        self.assertEqual(summary["requested_bit_budget"], 12)\n        self.assertEqual(summary["achieved_bit_budget"], 8)\n        self.assertEqual(summary["unused_bit_budget"], 4)\n        self.assertEqual(summary["achieved_channel_counts"],\n                         {"4": 1, "8": 2, "16": 0})\n        self.assertEqual(result["small"].metadata["requested_bit_budget"], 12)\n        self.assertEqual(result["small"].metadata["achieved_channel_counts"],\n                         {"4": 1, "8": 2, "16": 0})\n\n    def test_auto_allocator_validates_target_and_alignment(self):\n        losses = {"a": {4: [2, 2, 2], 8: [1, 1, 1], 16: [0, 0, 0]}}\n        with self.assertRaises(ValueError):\n            allocate_model_channels_auto(losses, 3.99)\n        with self.assertRaises(ValueError):\n            allocate_model_channels_auto(losses, 16.01)\n        with self.assertRaises(ValueError):\n            allocate_model_channels_auto(losses, 8, alignment=0)\n\n\nclass ThreeLevelReferenceTest(unittest.TestCase):\n\n    def test_activation_aware_losses_share_reference_and_fp16_is_zero(self):\n        torch.manual_seed(5)\n        activation = torch.randn(2, 3, 128)\n        weight = torch.randn(6, 128)\n        losses, awq = estimate_channel_losses(activation, weight)\n\n        self.assertEqual(set(losses), {4, 8, 16})\n        torch.testing.assert_close(losses[16], torch.zeros(6), rtol=0, atol=0)\n        self.assertEqual(awq.shape, (128,))\n        self.assertTrue((losses[4] >= 0).all() and (losses[8] >= 0).all())\n\n    def test_all_fp16_is_exact(self):\n        torch.manual_seed(7)\n        x = torch.randn(3, 128, dtype=torch.float32)\n        weight = torch.randn(4, 128, dtype=torch.float32)\n        allocation = allocate_channels(\n            {4: [2.0] * 4, 8: [1.0] * 4, 16: [0.0] * 4},\n            ThreeLevelBudget(0, 0, 100),\n        )\n\n        actual = fake_linear(x, weight, allocation)\n        expected = torch.nn.functional.linear(x, weight)\n        torch.testing.assert_close(actual, expected, rtol=0, atol=0)\n\n    def test_mixed_output_matches_per_partition_reference(self):\n        torch.manual_seed(11)\n        x = torch.randn(2, 128)\n        weight = torch.randn(4, 128)\n        allocation = ThreeLevelAllocation(\n            indices={4: (0, 1), 8: (2,), 16: (3,)},\n            scores={4: (0.0,) * 4, 8: (0.0,) * 4, 16: (0.0,) * 4},\n            budget=ThreeLevelBudget(50, 25, 25),\n        )\n\n        actual = fake_linear(x, weight, allocation)\n        self.assertEqual(actual.shape, (2, 4))\n        torch.testing.assert_close(actual[:, 3], x @ weight[3], rtol=0, atol=0)\n        self.assertTrue(torch.isfinite(actual).all())\n\n    def test_packed_module_matches_fake_reference(self):\n        torch.manual_seed(19)\n        x = torch.randn(3, 128)\n        weight = torch.randn(8, 128)\n        allocation = allocate_channels(\n            {4: torch.linspace(8, 1, 8),\n             8: torch.linspace(4, 0.5, 8),\n             16: torch.zeros(8)},\n            ThreeLevelBudget(50, 25, 25),\n        )\n        module = ThreeLevelLinear.from_weight(weight, allocation)\n\n        actual = module(x)\n        expected = fake_linear(x, weight, allocation)\n        torch.testing.assert_close(actual, expected, rtol=2e-3, atol=2e-3)\n\n    def test_packed_module_state_dict_round_trip(self):\n        weight = torch.randn(4, 128)\n        allocation = allocate_channels(\n            {4: [2.0] * 4, 8: [1.0] * 4, 16: [0.0] * 4},\n            ThreeLevelBudget(50, 25, 25),\n        )\n        original = ThreeLevelLinear.from_weight(weight, allocation)\n        original._sm75_int4_expanded = ("runtime-only", torch.empty(1))\n        loaded = ThreeLevelLinear(128, 4)\n        loaded._sm75_int4_expanded = ("stale", torch.empty(1))\n        loaded.load_state_dict(original.state_dict())\n\n        self.assertNotIn("_sm75_int4_expanded", original.state_dict())\n        self.assertIsNone(loaded._sm75_int4_expanded)\n        torch.testing.assert_close(loaded.dequantize_weight(), original.dequantize_weight())\n\n    def test_runtime_caches_clear_on_apply_and_load(self):\n        module = ThreeLevelLinear(128, 4)\n        module._sm75_fp16_placeholders = object()\n        module._sm75_int4_expanded = object()\n\n        module.to(dtype=torch.float16)\n\n        self.assertIsNone(module._sm75_fp16_placeholders)\n        self.assertIsNone(module._sm75_int4_expanded)\n\n    def test_backward_compatible_two_level_state_dict(self):\n        weight = torch.randn(4, 128)\n        allocation = allocate_channels(\n            {4: [2.0] * 4, 8: [1.0] * 4, 16: [0.0] * 4},\n            ThreeLevelBudget(50, 50, 0),\n        )\n        original = ThreeLevelLinear.from_weight(weight, allocation)\n        legacy = original.state_dict()\n        legacy.pop("weight_fp16")\n        legacy.pop("indices_16")\n        loaded = ThreeLevelLinear(128, 4)\n        loaded.load_state_dict(legacy, strict=True)\n\n        self.assertEqual(loaded.indices_16.numel(), 0)\n        torch.testing.assert_close(loaded.dequantize_weight(), original.dequantize_weight())\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'mixllm/test/test_runtime_capability.py': 'import unittest\n\nfrom mixllm.runtime_capability import RuntimeCapability\n\n\nclass RuntimeCapabilityTest(unittest.TestCase):\n\n    def test_t4_uses_reference_until_backend_is_validated(self):\n        self.assertEqual(RuntimeCapability(7, 5).select_backend(), "reference")\n\n    def test_auto_selects_validated_t4_backend(self):\n        self.assertEqual(RuntimeCapability(7, 5, sm75_available=True).select_backend(), "sm75")\n\n    def test_auto_selects_ampere_backend(self):\n        self.assertEqual(RuntimeCapability(8, 0).select_backend(), "ampere")\n\n    def test_unsupported_gpu_uses_reference(self):\n        self.assertEqual(RuntimeCapability(7, 0).select_backend(), "reference")\n\n    def test_rejects_forced_ampere_on_t4(self):\n        with self.assertRaises(RuntimeError):\n            RuntimeCapability(7, 5, "ampere").select_backend()\n\n    def test_rejects_unavailable_sm75_backend(self):\n        with self.assertRaises(RuntimeError):\n            RuntimeCapability(7, 5, "sm75").select_backend()\n\n    def test_rejects_unknown_backend(self):\n        with self.assertRaises(ValueError):\n            RuntimeCapability(8, 0, "unknown").select_backend()\n\n\nif __name__ == "__main__":\n    unittest.main()', 'mixllm/test/test_sm75_backend.py': 'import os\nimport unittest\nfrom unittest import mock\n\nimport torch\n\nfrom mixllm.nn.modules.three_level_linear import ThreeLevelLinear\nfrom mixllm.quantization.three_level import ThreeLevelAllocation, ThreeLevelBudget\nfrom mixllm import sm75_backend\nfrom mixllm.sm75_backend import (\n    load_sm75_backend,\n    quantize_activation,\n    quantize_activation_native,\n    quantized_reference,\n    three_level_linear,\n    three_level_linear_prequantized,\n)\n\n\n\nclass SM75PythonDispatchTest(unittest.TestCase):\n\n    def _module(self, counts):\n        n4, n8, n16 = counts\n        indices = {}\n        start = 0\n        for bit, count in ((4, n4), (8, n8), (16, n16)):\n            indices[bit] = tuple(range(start, start + count))\n            start += count\n        allocation = ThreeLevelAllocation(indices=indices, scores={bit: (0.0,) * start for bit in (4, 8, 16)}, budget=ThreeLevelBudget(100, 0, 0))\n        return ThreeLevelLinear.from_weight(torch.randn(start, 128), allocation)\n\n    def test_empty_rows_bypass_native_operators(self):\n        module = self._module((1, 1, 1))\n        with mock.patch.object(sm75_backend, \'_LOADED\', True), mock.patch.object(sm75_backend, \'quantize_activation_native\') as quantize, mock.patch.object(sm75_backend, \'three_level_linear_prequantized\') as gemm:\n            actual = three_level_linear(module, torch.empty(0, 128), torch)\n        self.assertEqual(actual.shape, (0, 3))\n        self.assertEqual(actual.dtype, torch.float32)\n        quantize.assert_not_called()\n        gemm.assert_not_called()\n\n    def test_zero_row_quantizers_return_consistent_shapes(self):\n        x = torch.empty(0, 256, dtype=torch.float16)\n        reference = quantize_activation(x, torch)\n        with mock.patch.object(sm75_backend, "_LOADED", True):\n            native = quantize_activation_native(x, torch)\n        for quantized, scales in (reference, native):\n            self.assertEqual(quantized.shape, (0, 256))\n            self.assertEqual(scales.shape, (2, 0))\n\n    def test_prefill_int4_expansion_is_signed_cached_and_invalidated(self):\n        module = self._module((2, 0, 0))\n        x = torch.empty(2, 128, dtype=torch.float16)\n\n        first = sm75_backend._expanded_int4_for_prefill(module, x, torch)\n        codes = torch.empty(2, 128, dtype=torch.uint8)\n        codes[:, 0::2] = module.weight_int4 & 0x0f\n        codes[:, 1::2] = module.weight_int4 >> 4\n        expected = (\n            codes.to(torch.int16)\n            - module.zero_int4.repeat_interleave(128, dim=1).to(torch.int16)\n        ).to(torch.int8)\n        torch.testing.assert_close(first, expected, rtol=0, atol=0)\n\n        second = sm75_backend._expanded_int4_for_prefill(module, x, torch)\n        self.assertIs(first, second)\n        module.weight_int4[0, 0] ^= 0x0f\n        third = sm75_backend._expanded_int4_for_prefill(module, x, torch)\n        self.assertIsNot(first, third)\n\n    def test_decode_keeps_packed_int4_and_does_not_expand(self):\n        module = self._module((2, 0, 0))\n        placeholder = sm75_backend._expanded_int4_for_prefill(\n            module, torch.empty(1, 128, dtype=torch.float16), torch,\n        )\n\n        self.assertEqual(placeholder.shape, (0, 128))\n        self.assertEqual(placeholder.data_ptr(), module.weight_int8.data_ptr())\n        self.assertIsNone(module._sm75_int4_expanded)\n\n    def test_rejects_malformed_partitions_and_invalidates_cache(self):\n        module = self._module((1, 1, 1))\n        x = torch.randn(2, 128, dtype=torch.float16)\n        input_int8 = torch.empty_like(x, dtype=torch.int8)\n        scale_act = torch.empty(1, 2, dtype=torch.float16)\n        malformed = (\n            ((0,), (0,), (2,)),\n            ((0,), (1,), ()),\n            ((0,), (1,), (3,)),\n        )\n        with mock.patch.object(sm75_backend, "_LOADED", True):\n            for indices in malformed:\n                with self.subTest(indices=indices):\n                    module.indices_4 = torch.tensor(indices[0], dtype=torch.int32)\n                    module.indices_8 = torch.tensor(indices[1], dtype=torch.int32)\n                    module.indices_16 = torch.tensor(indices[2], dtype=torch.int32)\n                    with self.assertRaisesRegex(ValueError, "complete output partition"):\n                        three_level_linear_prequantized(\n                            module, x, input_int8, scale_act, torch,\n                        )\n\n    def test_partition_validation_cache_tracks_in_place_mutation(self):\n        module = self._module((1, 1, 1))\n        x = torch.randn(2, 128, dtype=torch.float16)\n        input_int8 = torch.empty_like(x, dtype=torch.int8)\n        scale_act = torch.empty(1, 2, dtype=torch.float16)\n        operator = mock.Mock(return_value=torch.empty(2, 3))\n        with mock.patch.object(sm75_backend, "_LOADED", True), mock.patch.object(\n                torch.ops.mixllm_sm75, "_three_level_linear_v2_unchecked",\n                operator, create=True):\n            three_level_linear_prequantized(\n                module, x, input_int8, scale_act, torch,\n            )\n            module.indices_8.copy_(module.indices_4)\n            with self.assertRaisesRegex(ValueError, "duplicates"):\n                three_level_linear_prequantized(\n                    module, x, input_int8, scale_act, torch,\n                )\n\n    def test_validates_prequantized_tensor_devices(self):\n        module = self._module((1, 1, 1))\n        x = torch.randn(2, 128, dtype=torch.float16)\n        input_int8 = torch.empty(2, 128, device="meta", dtype=torch.int8)\n        scale_act = torch.empty(1, 2, dtype=torch.float16)\n        with mock.patch.object(sm75_backend, "_LOADED", True):\n            with self.assertRaisesRegex(ValueError, "all operator tensors"):\n                three_level_linear_prequantized(\n                    module, x, input_int8, scale_act, torch,\n                )\n    def test_pure_fp16_skips_quantization_and_reuses_placeholders(self):\n        module = self._module((0, 0, 3))\n        x = torch.randn(2, 128, dtype=torch.float16)\n        with mock.patch.object(sm75_backend, \'_LOADED\', True), mock.patch.object(sm75_backend, \'quantize_activation_native\') as quantize, mock.patch.object(sm75_backend, \'three_level_linear_prequantized\', return_value=torch.empty(2, 3)) as gemm:\n            three_level_linear(module, x, torch)\n            first = gemm.call_args.args[2:4]\n            three_level_linear(module, x, torch)\n            second = gemm.call_args.args[2:4]\n        quantize.assert_not_called()\n        self.assertIs(first[0], second[0])\n        self.assertIs(first[1], second[1])\n\n    def test_quantized_and_mixed_partitions_quantize_once(self):\n        for counts in ((3, 0, 0), (0, 3, 0), (1, 1, 1), (1, 0, 2)):\n            module = self._module(counts)\n            x = torch.randn(2, 128, dtype=torch.float16)\n            quantized = torch.empty_like(x, dtype=torch.int8)\n            scales = torch.empty(1, 2, dtype=torch.float16)\n            with mock.patch.object(sm75_backend, \'_LOADED\', True), mock.patch.object(sm75_backend, \'quantize_activation_native\', return_value=(quantized, scales)) as quantize, mock.patch.object(sm75_backend, \'three_level_linear_prequantized\', return_value=torch.empty(2, 3)) as gemm:\n                three_level_linear(module, x, torch)\n            quantize.assert_called_once_with(x, torch, 128)\n            self.assertIs(gemm.call_args.args[2], quantized)\n            self.assertIs(gemm.call_args.args[3], scales)\n\n@unittest.skipUnless(\n    os.environ.get("MIXLLM_TEST_SM75") == "1" and torch.cuda.is_available(),\n    "requires an explicit SM75 GPU test run",\n)\nclass SM75BackendTest(unittest.TestCase):\n\n    @classmethod\n    def setUpClass(cls):\n        if tuple(torch.cuda.get_device_capability()) != (7, 5):\n            raise unittest.SkipTest("requires compute capability 7.5")\n        load_sm75_backend(torch)\n\n    def _run_case(self, counts, rows=3, width=128, seed=31):\n        torch.manual_seed(seed)\n        n4, n8, n16 = counts\n        output_width = sum(counts)\n        order = torch.randperm(output_width).tolist()\n        allocation = ThreeLevelAllocation(\n            indices={\n                4: tuple(sorted(order[:n4])),\n                8: tuple(sorted(order[n4:n4 + n8])),\n                16: tuple(sorted(order[n4 + n8:])),\n            },\n            scores={bit: (0.0,) * output_width for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(100, 0, 0),\n        )\n        weight = torch.randn(output_width, width, device="cuda", dtype=torch.float16)\n        x = torch.randn(rows, width, device="cuda", dtype=torch.float16)\n        module = ThreeLevelLinear.from_weight(weight, allocation).cuda()\n\n        actual = three_level_linear(module, x, torch)\n        expected = quantized_reference(module, x, torch)\n        torch.testing.assert_close(actual, expected, rtol=2e-2, atol=2e-2)\n        self.assertTrue(torch.isfinite(actual).all())\n\n    def test_native_activation_quantizer_matches_reference(self):\n        for rows, width in ((1, 128), (5, 512), (32, 1024)):\n            with self.subTest(rows=rows, width=width):\n                torch.manual_seed(rows * 1000 + width)\n                x = torch.randn(rows, width, device="cuda", dtype=torch.float16)\n                actual_q, actual_scale = quantize_activation_native(x, torch)\n                expected_q, expected_scale = quantize_activation(x, torch)\n                torch.testing.assert_close(actual_scale, expected_scale, rtol=2e-3,\n                                           atol=2e-5)\n                self.assertLessEqual(\n                    int((actual_q.to(torch.int16) - expected_q.to(torch.int16))\n                        .abs().max().item()),\n                    1,\n                )\n\n    def test_mixed_and_empty_partitions(self):\n        for counts in ((5, 3, 2), (10, 0, 0), (0, 10, 0), (0, 0, 10),\n                       (0, 4, 6), (7, 0, 3)):\n            with self.subTest(counts=counts):\n                self._run_case(counts)\n\n    def test_int4_tile_width_boundaries(self):\n        for channels in (1, 15, 16, 17, 63, 64, 65):\n            with self.subTest(channels=channels):\n                self._run_case((channels, 0, 0), rows=3,\n                               seed=7000 + channels)\n    def test_random_rows_widths_and_determinism(self):\n        for rows, width in ((1, 128), (5, 128), (3, 256), (7, 384)):\n            with self.subTest(rows=rows, width=width):\n                self._run_case((4, 3, 3), rows=rows, width=width,\n                               seed=rows * 1000 + width)\n\n        allocation = ThreeLevelAllocation(\n            indices={4: (1,), 8: (2,), 16: (0,)},\n            scores={bit: (0.0,) * 3 for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(100, 0, 0),\n        )\n        weight = torch.randn(3, 128, device="cuda", dtype=torch.float16)\n        x = torch.randn(3, 128, device="cuda", dtype=torch.float16)\n        module = ThreeLevelLinear.from_weight(weight, allocation).cuda()\n        first = three_level_linear(module, x, torch)\n        second = three_level_linear(module, x, torch)\n        torch.testing.assert_close(first, second, rtol=0, atol=0)\n\n    def test_non_default_stream_dependency(self):\n        allocation = ThreeLevelAllocation(\n            indices={4: (1,), 8: (2,), 16: (0,)},\n            scores={bit: (0.0,) * 3 for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(34, 33, 33),\n        )\n        module = ThreeLevelLinear.from_weight(\n            torch.randn(3, 128, device="cuda", dtype=torch.float16), allocation,\n        ).cuda()\n        producer = torch.cuda.Stream()\n        consumer = torch.cuda.current_stream()\n        with torch.cuda.stream(producer):\n            x = torch.randn(5, 128, device="cuda", dtype=torch.float16)\n            ready = torch.cuda.Event()\n            ready.record()\n        consumer.wait_event(ready)\n        actual = three_level_linear(module, x, torch)\n        expected = quantized_reference(module, x, torch)\n        torch.testing.assert_close(actual, expected, rtol=2e-2, atol=2e-2)\n\n    def test_cuda_graph_capture(self):\n        allocation = ThreeLevelAllocation(\n            indices={4: (0, 3), 8: (1,), 16: (2,)},\n            scores={bit: (0.0,) * 4 for bit in (4, 8, 16)},\n            budget=ThreeLevelBudget(50, 25, 25),\n        )\n        module = ThreeLevelLinear.from_weight(\n            torch.randn(4, 128, device="cuda", dtype=torch.float16), allocation,\n        ).cuda()\n        static_input = torch.randn(2, 128, device="cuda", dtype=torch.float16)\n        side_stream = torch.cuda.Stream()\n        side_stream.wait_stream(torch.cuda.current_stream())\n        with torch.cuda.stream(side_stream):\n            for _ in range(3):\n                three_level_linear(module, static_input, torch)\n        torch.cuda.current_stream().wait_stream(side_stream)\n        graph = torch.cuda.CUDAGraph()\n        with torch.cuda.graph(graph):\n            captured = three_level_linear(module, static_input, torch)\n        graph.replay()\n        expected = quantized_reference(module, static_input, torch)\n        torch.testing.assert_close(captured, expected, rtol=2e-2, atol=2e-2)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'mixllm/test/test_sm75_source.py': 'from pathlib import Path\nimport unittest\n\n\nclass SM75SourceContractTest(unittest.TestCase):\n\n    def test_one_gemm_kernel_contains_integer_and_fp16_tensorcore_paths(self):\n        source = Path(__file__).parents[1] / "kernels" / "three_level_sm75.cu"\n        text = source.read_text(encoding="utf-8")\n        self.assertEqual(text.count("three_level_tensorcore_kernel<<<"), 1)\n        self.assertEqual(text.count("three_level_decode_kernel<<<"), 1)\n        self.assertIn("quantize_activation_sm75_kernel", text)\n        self.assertIn("three_level_tensorcore_kernel", text)\n        self.assertIn("three_level_decode_kernel", text)\n        self.assertIn("signed char", text)\n        self.assertIn("input_int8", text)\n        self.assertIn("packed_int4", text)\n        self.assertIn("weight_fp16", text)\n        self.assertIn("wmma::mma_sync", text)\n        self.assertIn("__dp4a", text)\n        self.assertIn("if (rows == 1)", text)\n        self.assertIn("constexpr int kPrefillWarps = 4", text)\n        self.assertIn("constexpr int kDecodeChannelsPerWarp = 4", text)\n        self.assertIn("constexpr int kDecodeSubwarp", text)\n        self.assertIn("three_level_tensorcore_reuse_kernel", text)\n        self.assertNotIn("three_level_tensorcore_reuse_kernel<<<", text)\n        self.assertIn("expanded_int4", text)\n        self.assertIn("expanded_int4.data_ptr<int8_t>()", text)\n        self.assertIn("expand_int4_sm75_kernel", text)\n        self.assertIn("three_level_linear_v2", text)\n        self.assertIn("three_level_linear_legacy_cuda", text)\n        self.assertIn("check_same_device", text)\n        self.assertIn("output_width > 0", text)\n        self.assertIn("width / 2", text)\n        self.assertIn("kPrefillWarps * kWarpSize", text)\n        self.assertIn("linear += blockDim.x", text)\n        self.assertIn("__syncthreads()", text)\n        self.assertIn("__half22float2(__hmul2(input2[k2], weights2[k2]))", text)\n        self.assertIn("reinterpret_cast<uint32_t*>(quantized", text)\n        decode = text.split("__global__ void three_level_decode_kernel", 1)[1]\n        decode = decode.split("void check_cuda_contiguous", 1)[0]\n        self.assertEqual(decode.count("__shfl_down_sync"), 3)\n        prefill = text.split("__global__ void three_level_tensorcore_kernel", 1)[1]\n        prefill = prefill.split("three_level_tensorcore_reuse_kernel", 1)[0]\n        self.assertNotIn("packed_int4", prefill)\n        self.assertNotIn("zero_int4", prefill)\n        self.assertIn("channels_per_block", text)\n        self.assertNotIn("three_level_partition_kernel", text)\n        self.assertNotIn("packed_weight_to_fp16", text)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'mixllm/test/test_model_gate.py': 'import importlib\nimport unittest\nfrom types import SimpleNamespace\n\nimport torch\nfrom torch import nn\n\n\nclass TinyBlock(nn.Module):\n\n    def __init__(self):\n        super().__init__()\n        self.proj = nn.Linear(4, 4, bias=False)\n\n    def forward(self, hidden):\n        return torch.tanh(self.proj(hidden))\n\n\nclass TinyCausalLM(nn.Module):\n\n    def __init__(self):\n        super().__init__()\n        self.embed = nn.Embedding(8, 4)\n        self.model = nn.Module()\n        self.model.layers = nn.ModuleList([TinyBlock()])\n        self.head = nn.Linear(4, 8, bias=False)\n\n    def forward(self, input_ids, labels=None, use_cache=False):\n        hidden = self.embed(input_ids)\n        for layer in self.model.layers:\n            hidden = layer(hidden)\n        logits = self.head(hidden)\n        loss = None\n        if labels is not None:\n            loss = nn.functional.cross_entropy(\n                logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))\n        return SimpleNamespace(logits=logits, loss=loss)\n\n\nclass ModelGateTest(unittest.TestCase):\n\n    def test_model_gate_module_imports(self):\n        module = importlib.import_module("mixllm.model_gate")\n        self.assertTrue(callable(module.run_model_gate))\n\n    def test_auto_model_gate_quantizes_and_reports_budget(self):\n        from mixllm.model_gate import run_model_gate\n        from mixllm.nn.modules.three_level_linear import ThreeLevelLinear\n\n        torch.manual_seed(13)\n        model = TinyCausalLM()\n        ids = torch.tensor([[0, 1, 2, 3]])\n\n        result = run_model_gate(\n            "tiny", None, model, ids, ids, target_average_bits=8,\n            group_size=4, calibration_rows=4)\n\n        self.assertEqual(result["packed_layers"], 1)\n        self.assertIsInstance(model.model.layers[0].proj, ThreeLevelLinear)\n        self.assertEqual(result["allocation_summary"]["target_average_bits"], 8)\n        self.assertLessEqual(result["average_weight_bits"], 8)\n        self.assertTrue(result["deterministic"])\n        self.assertTrue(result["finite"])\n\n    def test_fixed_budget_model_gate_remains_supported(self):\n        from mixllm.model_gate import run_model_gate\n        from mixllm.quantization.three_level import ThreeLevelBudget\n\n        torch.manual_seed(17)\n        model = TinyCausalLM()\n        ids = torch.tensor([[0, 1, 2, 3]])\n        result = run_model_gate(\n            "tiny", None, model, ids, ids,\n            budget=ThreeLevelBudget(50, 25, 25), group_size=4,\n            calibration_rows=4)\n\n        self.assertEqual(result["channel_counts"], {"4": 2, "8": 1, "16": 1})\n        self.assertEqual(\n            result["allocation_summary"]["allocator"],\n            "fixed_precision_percentages")\n\n    def test_model_gate_uses_eval_and_restores_training_mode(self):\n        from mixllm.model_gate import run_model_gate\n\n        model = TinyCausalLM().train()\n        ids = torch.tensor([[0, 1, 2, 3]])\n        observed_modes = []\n        handle = model.register_forward_pre_hook(\n            lambda current, args, kwargs: observed_modes.append(current.training),\n            with_kwargs=True,\n        )\n        try:\n            run_model_gate(\n                "tiny", None, model, ids, ids, target_average_bits=8,\n                group_size=4, calibration_rows=4)\n        finally:\n            handle.remove()\n\n        self.assertTrue(observed_modes)\n        self.assertFalse(any(observed_modes))\n        self.assertTrue(model.training)\n\n    def test_model_gate_requires_exactly_one_budget_mode(self):\n        from mixllm.model_gate import run_model_gate\n        from mixllm.quantization.three_level import ThreeLevelBudget\n\n        ids = torch.tensor([[0, 1]])\n        with self.assertRaises(ValueError):\n            run_model_gate("tiny", None, TinyCausalLM(), ids, ids, group_size=4)\n        with self.assertRaises(ValueError):\n            run_model_gate(\n                "tiny", None, TinyCausalLM(), ids, ids,\n                budget=ThreeLevelBudget(100, 0, 0), target_average_bits=8,\n                group_size=4)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'mixllm/test/test_vllm_three_level.py': 'import unittest\n\nfrom mixllm.vllm_three_level import (\n    PINNED_VLLM_COMMIT,\n    VLLMThreeLevelConfig,\n    remap_partition_indices_for_tp,\n    restore_global_partition_indices,\n    validate_partition_indices,\n)\n\n\nclass VLLMThreeLevelContractTest(unittest.TestCase):\n\n    def test_pins_real_upstream_submodule_commit(self):\n        self.assertEqual(PINNED_VLLM_COMMIT,\n                         "5fbbfe9a4c13094ad72ed3d6b4ef208a7ddc0fd7")\n\n    def test_config_and_backend_gate(self):\n        config = VLLMThreeLevelConfig.from_quantization_config({\n            "quant_method": "mixllm_three_level",\n            "precision_percentages": {"4": 75, "8": 20, "16": 5},\n            "group_size": 128,\n        })\n        self.assertEqual(config.select_backend((7, 5)), "reference")\n        self.assertEqual(config.select_backend((8, 0)), "ampere")\n\n    def test_requires_three_level_percentages(self):\n        with self.assertRaises(ValueError):\n            VLLMThreeLevelConfig.from_quantization_config({"quant_method": "mixllm"})\n\n    def test_partition_validation(self):\n        validate_partition_indices({4: [2, 3], 8: [1], 16: [0]}, 4)\n        with self.assertRaises(ValueError):\n            validate_partition_indices({4: [1], 8: [1], 16: [0]}, 3)\n\n    def test_tensor_parallel_index_remapping_round_trip(self):\n        global_indices = {4: [0, 3, 6, 7], 8: [1, 5], 16: [2, 4]}\n        local = remap_partition_indices_for_tp(global_indices, 8, 2, 4)\n        self.assertEqual(local, {4: (1,), 8: (3,), 16: (0, 2)})\n        restored = restore_global_partition_indices(local, 2, 4)\n        self.assertEqual(restored, {4: (3,), 8: (5,), 16: (2, 4)})\n\n    def test_tensor_parallel_rejects_invalid_shards(self):\n        indices = {4: [0, 1], 8: [2], 16: [3]}\n        with self.assertRaises(ValueError):\n            remap_partition_indices_for_tp(indices, 4, 3, 2)\n\n\nif __name__ == "__main__":\n    unittest.main()'}
source_manifest = {'algorithm': 'sha256', 'source_sha256': '5db9c0a998d848f5a99cfbbdcb858dd4104c047862972f488011342b7e0f41c0', 'files': {'mixllm/__init__.py': 'f603b78d313e22c460c3d53eea26d010b78a2ef56715b6aac73976be082989c0', 'mixllm/quantization/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mixllm/nn/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mixllm/nn/modules/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'mixllm/quantization/three_level.py': '1f618fa6a8a19989c0e4c2438431cba3032eb1bc6193c4aafcc99af6420864a8', 'mixllm/nn/modules/mixllm_config.py': '9b9d498fa7b0ca60ccaf687999e797df4aa72a99db4fd5e2df0f21af9f77e73e', 'mixllm/nn/modules/three_level_linear.py': '3b004caa673d4ca071517c1a26f3459dc20679f3d8df6b42e1235c6fcf295a12', 'mixllm/nn/modules/ops.py': 'cb24d8c8dbc600550bfde3b3299eb6e7962eb11ff5124581cae53d11ea161b72', 'mixllm/runtime_capability.py': 'b812981827869ddd84763000c121767515668eeaf8618b0c894f3e5f7426c997', 'mixllm/sm75_backend.py': '9a9b78f7fa80bca9e131e5a2ca6e535f72d6eef9a10092b2576a63a49baa6706', 'mixllm/model_gate.py': '046890ce9b3829be6f7b092f06f74478780a5270f53c666d989dd449b16d1dbc', 'mixllm/vllm_three_level.py': 'a31a60e5837bab401501a2b21dccb70de8c95323caa201e1c7fc96fb14fde3ce', 'mixllm/kernels/three_level_sm75.cu': '66b2fcaac61b43b2ad2791830df98573767bb2300c2db824b3fd15148bc988a2', 'mixllm/test/test_three_level.py': '5f86e1ab1c9c9be49121ea56d2380483e7105cb114c81e92b6c5e564847afaa3', 'mixllm/test/test_runtime_capability.py': '344d9193b1af345aeb47d2ada0600613d5ee779f831d4b1f776a1ef241d01bcd', 'mixllm/test/test_sm75_backend.py': '24c51f8f1ce82e3e8c75652a0e469c043eef5eaccd8dfc7877b1b3a57e65a069', 'mixllm/test/test_sm75_source.py': '5becd8b39a8b9899b1255ea5530da21222c078c22abef741ce52494a42952350', 'mixllm/test/test_model_gate.py': '2c34de198083b2d27f4b703e16c128a6dc5f1bd2e2f8b9b9a87fae726879f5e8', 'mixllm/test/test_vllm_three_level.py': '7bc58095c58639546622210a4f7337bca2dd6b64f0216b471da37b7b25670efc'}, 'workspace_commit': '0a14e8bd86fb04dd962e402b7eff69154e9baaa8', 'mixllm_commit': 'ee7a0d4f2e97079fcc65fc6c5a946fe688b46d85', 'workspace_dirty': True, 'mixllm_dirty': True}
root = ARTIFACT_DIR / 'mixllm-3level'
for relative, text in sources.items():
    path = root / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    assert hashlib.sha256(path.read_bytes()).hexdigest() == source_manifest['files'][relative]
digest = hashlib.sha256()
for relative in sorted(sources): digest.update(relative.encode() + b'\0' + sources[relative].encode() + b'\0')
assert digest.hexdigest() == source_manifest['source_sha256']
source_manifest['embedded_file_count'] = len(sources)
(ARTIFACT_DIR / 'mixllm_3level_source_manifest.json').write_text(json.dumps(source_manifest, indent=2, sort_keys=True))
sys.path.insert(0, str(root))
print('Embedded source SHA-256:', source_manifest['source_sha256'])


In [ ]:
cuda_available = torch.cuda.is_available()
capability = tuple(torch.cuda.get_device_capability(0)) if cuda_available else None
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
is_t4 = capability == (7, 5) and gpu_name and 'T4' in gpu_name.upper()
report = {'schema_version': 4, 'target': 'NVIDIA T4 / SM75', 'provenance': source_manifest,
          'environment': {'cuda_available': cuda_available, 'gpu_name': gpu_name, 'capability': capability,
                          'torch_version': torch.__version__, 'python_version': platform.python_version()},
          'gates': {'t4_hardware': 'passed' if is_t4 else 'failed',
                    'full_model_qwen_quality': 'not_run', 'full_model_qwen_throughput': 'not_run'},
          'claims': {'benchmark_scope': 'native_operator_microbenchmark', 'full_model_qwen_quality_claimed': False}}


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(root) + os.pathsep + test_env.get('PYTHONPATH', '')
if capability == (7, 5): test_env['MIXLLM_TEST_SM75'] = '1'
tests = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(root / 'mixllm/test'), '-p', 'test_three_level.py'], cwd=root, env=test_env, text=True, capture_output=True, timeout=600)
print(tests.stdout); print(tests.stderr)
report['tests'] = {'returncode': tests.returncode, 'model_gate_test_embedded': 'mixllm/test/test_model_gate.py' in sources}
report['gates']['embedded_contract_tests'] = 'passed' if tests.returncode == 0 else 'failed'


In [ ]:
from mixllm.model_gate import run_model_gate
from mixllm.quantization.three_level import ThreeLevelBudget, allocate_channels, allocate_model_channels, allocate_model_channels_auto, estimate_channel_losses
assert callable(run_model_gate)
torch.manual_seed(1234); x = torch.randn(4, 3, 128); w = torch.randn(8, 128)
losses, awq_stat = estimate_channel_losses(x, w)
allocation = allocate_channels(losses, ThreeLevelBudget(50, 25, 25)); allocation.verify(w.shape[0])
fixed = allocate_model_channels({'layer': losses}, ThreeLevelBudget(50, 25, 25))
automatic, allocation_summary = allocate_model_channels_auto({'layer': losses}, 8.0)
fixed['layer'].verify(w.shape[0]); automatic['layer'].verify(w.shape[0])
assert allocation_summary['achieved_average_bits'] <= 8.0 and torch.isfinite(awq_stat).all()
report['gates'].update(model_gate_import='passed', fixed_allocator='passed', auto_allocator='passed')
report['allocator_check'] = allocation_summary


In [ ]:
benchmarks = {'status': 'not_run', 'baseline': 'torch_fp16_linear', 'scenarios': {}}
if capability == (7, 5):
    from mixllm.nn.modules.three_level_linear import ThreeLevelLinear
    from mixllm.quantization.three_level import ThreeLevelAllocation, ThreeLevelBudget
    from mixllm.sm75_backend import benchmark_sm75_backend, load_sm75_backend
    load_sm75_backend(torch)
    def make_case(n, width, counts, rows):
        n4, n8, n16 = counts; assert n4 + n8 + n16 == n
        alloc = ThreeLevelAllocation(indices={4: tuple(range(n4)), 8: tuple(range(n4, n4+n8)), 16: tuple(range(n4+n8, n))}, scores={b: (0.0,) * n for b in (4, 8, 16)}, budget=ThreeLevelBudget(*(100*c/n for c in counts)))
        packed = ThreeLevelLinear.from_weight(torch.randn(n, width, device='cuda', dtype=torch.float16), alloc).cuda()
        return benchmark_sm75_backend(packed, rows=rows, torch_module=torch, warmup=10, iterations=50)
    cases = {'smoke_mixed_4_8_16': (96, 512, (64, 24, 8), (1, 8, 32, 128)), 'qwen_qkv_mixed_4_8_16': (3584, 3584, (2400, 896, 288), (1, 16, 128)), 'qwen_qkv_pure_int4': (3584, 3584, (3584, 0, 0), (1, 16, 128)), 'qwen_qkv_pure_int8': (3584, 3584, (0, 3584, 0), (1, 16, 128)), 'qwen_qkv_pure_fp16': (3584, 3584, (0, 0, 3584), (1, 16, 128))}
    benchmarks['scenarios'] = {name: make_case(*args) for name, args in cases.items()}
    benchmarks['status'] = 'measured'
report['benchmarks'] = benchmarks
(ARTIFACT_DIR / 'mixllm_3level_benchmarks.json').write_text(json.dumps(benchmarks, indent=2, sort_keys=True))


In [ ]:
gates = report['gates']
if benchmarks['status'] == 'measured':
    shapes = [shape for case in benchmarks['scenarios'].values() for shape in case['shapes']]
    mixed = benchmarks['scenarios']['qwen_qkv_mixed_4_8_16']['shapes']
    correctness = all(s['max_abs_error'] <= 0.15 for s in shapes)
    decode_gemm = all(s['gemm_p50_ratio_vs_dense'] <= 1.05 for s in mixed if s['rows'] == 1)
    decode_e2e = all(s['end_to_end_p50_ratio_vs_dense'] <= 1.05 for s in mixed if s['rows'] == 1)
    prefill_e2e = all(s['end_to_end_p50_ratio_vs_dense'] <= 1.05 for s in mixed if s['rows'] > 1)
    gates.update(sm75_native_benchmarks='passed', sm75_native_correctness='passed' if correctness else 'failed', mixed_decode_gemm_performance='passed' if decode_gemm else 'failed', mixed_decode_end_to_end_performance='passed' if decode_e2e else 'failed', mixed_prefill_end_to_end_performance='passed' if prefill_e2e else 'failed')
    production_ready = correctness and decode_e2e and prefill_e2e
else:
    gates.update(sm75_native_benchmarks='not_run', sm75_native_correctness='not_run'); production_ready = False
print('TESTS_RETURNCODE', tests.returncode, flush=True); print('TESTS_STDOUT_TAIL', tests.stdout[-2000:], flush=True); print('TESTS_STDERR_TAIL', tests.stderr[-2000:], flush=True); print('IS_T4', is_t4, flush=True); print('GATES_PRE_EXEC', gates, flush=True); print('BENCHMARKS_PRE_EXEC', benchmarks, flush=True); execution = bool(is_t4 and tests.returncode == 0 and gates.get('model_gate_import') == 'passed' and benchmarks['status'] == 'measured')
report['gate_status'] = {'execution': 'passed' if execution else 'failed', 't4_production': 'passed' if production_ready else 'failed', 'terminal_decision': 'go' if production_ready else 'no_go', 'reason': 'all native correctness and mixed end-to-end gates passed' if production_ready else 'production requires native correctness plus mixed decode and prefill end-to-end performance'}
(ARTIFACT_DIR / 'mixllm_3level_gate.json').write_text(json.dumps(report, indent=2, sort_keys=True))
print(json.dumps(report['gate_status'], indent=2)); print('Full-model Qwen quality:', gates['full_model_qwen_quality'])
assert execution, 'T4 gate did not execute completely; inspect artifact'
